# CMPT 354 Mini-Project — Steps 4 and 5

This notebook creates `library.db`, converts the final E/R model into SQLite tables, adds constraints and triggers, and populates every table with at least 10 realistic tuples.

## Main integrity requirements implemented

- Primary, candidate, and foreign keys
- Valid status and category domains through `CHECK`
- One active loan per borrowable item
- Maximum five active loans per member
- Only active members may borrow
- Unpaid fines block new loans and event registration
- Loan-duration limits by item type
- Fine maximums by item category
- Room-capacity and room-overlap checks
- Event-capacity checks
- Employee-role authorization for loans, event organization, donations, and acquisitions
- Event registration capacity and duplicate-registration prevention

Run the notebook from top to bottom. The database file will be created in the same folder as the notebook.


## 0. Load JupySQL

In [1]:
%load_ext sql

## 1. Create and connect to the SQLite database

In [2]:
%sql sqlite:///library.db

Connecting to 'sqlite:///library.db'

Enable SQLite foreign-key enforcement for this connection.

In [3]:
%%sql
PRAGMA foreign_keys = ON;

Running query in 'sqlite:///library.db'

++
||
++
++

## 2. Remove old tables when rerunning the notebook

In [4]:
%%sql

DROP VIEW IF EXISTS ExpectedFine;

DROP TABLE IF EXISTS AddCollection;
DROP TABLE IF EXISTS HelpRequest;
DROP TABLE IF EXISTS Auth;

DROP TABLE IF EXISTS EventRegistration;
DROP TABLE IF EXISTS VolunteerAssignment;
DROP TABLE IF EXISTS EventAudience;
DROP TABLE IF EXISTS EventInterest;
DROP TABLE IF EXISTS OrganizedBy;
DROP TABLE IF EXISTS Event;
DROP TABLE IF EXISTS Room;

DROP TABLE IF EXISTS Fine;
DROP TABLE IF EXISTS Loan;
DROP TABLE IF EXISTS BorrowableItem;
DROP TABLE IF EXISTS LibraryItem;

DROP TABLE IF EXISTS DonatedItems;
DROP TABLE IF EXISTS Donation;
DROP TABLE IF EXISTS FutureAcquisitions;

DROP TABLE IF EXISTS PersonInterest;
DROP TABLE IF EXISTS Audience;
DROP TABLE IF EXISTS Employee;
DROP TABLE IF EXISTS Interest;
DROP TABLE IF EXISTS Member;
DROP TABLE IF EXISTS Person;

Running query in 'sqlite:///library.db'

++
||
++
++

## 3. Create the tables

### Creating the Person table

In [5]:
%%sql

CREATE TABLE Person (
    personID INTEGER PRIMARY KEY,
    firstName TEXT NOT NULL,
    lastName TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    phoneNumber TEXT,
    address TEXT,
    dateOfBirth TEXT,

    CHECK (
        dateOfBirth IS NULL
        OR (
            dateOfBirth GLOB
                '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
            AND date(julianday(dateOfBirth)) = dateOfBirth
        )
    )
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Member table

In [6]:
%%sql

CREATE TABLE Member (
    memberID INTEGER PRIMARY KEY,
    personID INTEGER NOT NULL UNIQUE,
    membershipDate TEXT NOT NULL,
    membershipStatus TEXT NOT NULL
        CHECK (
            membershipStatus IN (
                'Active',
                'Inactive',
                'Suspended'
            )
        ),

    CHECK (
        membershipDate GLOB
            '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
        AND date(julianday(membershipDate)) = membershipDate
    ),

    FOREIGN KEY (personID)
        REFERENCES Person(personID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Interest table

In [7]:
%%sql
CREATE TABLE Interest (
    interestID INTEGER PRIMARY KEY,
    interestName TEXT NOT NULL UNIQUE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Employee table

In [8]:
%%sql

CREATE TABLE Employee (
    employeeID INTEGER PRIMARY KEY,
    personID INTEGER NOT NULL UNIQUE,

    position TEXT NOT NULL
        CHECK (
            position IN (
                'Librarian',
                'Library Assistant',
                'Event Coordinator',
                'Manager',
                'Technician'
            )
        ),

    hireDate TEXT NOT NULL,

    salary REAL NOT NULL
        CHECK (salary >= 0),

    employeeStatus TEXT NOT NULL
        CHECK (
            employeeStatus IN (
                'Active',
                'Inactive',
                'On Leave'
            )
        ),

    CHECK (
        hireDate GLOB
            '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
        AND date(julianday(hireDate)) = hireDate
    ),

    FOREIGN KEY (personID)
        REFERENCES Person(personID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the LibraryItem table

In [9]:
%%sql

CREATE TABLE LibraryItem (
    itemID INTEGER PRIMARY KEY,
    itemTitle TEXT NOT NULL,

    itemType TEXT NOT NULL
        CHECK (
            itemType IN (
                'Print Book',
                'Online Book',
                'Magazine',
                'Scientific Journal',
                'Record',
                'Laptop',
                'Charger',
                'Marker',
                'Eraser'
            )
        ),

    author TEXT,
    publisher TEXT,

    publicationYear INTEGER
        CHECK (
            publicationYear IS NULL
            OR publicationYear > 0
        ),

    language TEXT,
    ISBN TEXT UNIQUE,
    onlineAccessURL TEXT,

    CHECK (
        itemType <> 'Online Book'
        OR onlineAccessURL IS NOT NULL
    )
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the BorrowableItem table

In [10]:
%%sql
CREATE TABLE BorrowableItem (
    borrowableItemID INTEGER PRIMARY KEY,
    itemID INTEGER NOT NULL,
    barcode TEXT NOT NULL UNIQUE,
    shelfLocation TEXT,
    itemCondition TEXT NOT NULL
        CHECK (itemCondition IN ('New', 'Good', 'Fair', 'Damaged')),
    itemStatus TEXT NOT NULL
        CHECK (itemStatus IN ('Available', 'Borrowed', 'Lost', 'Maintenance')),
    FOREIGN KEY (itemID) REFERENCES LibraryItem(itemID)
        ON UPDATE CASCADE ON DELETE RESTRICT
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Loan table

In [11]:
%%sql

CREATE TABLE Loan (
    loanID INTEGER PRIMARY KEY,
    memberID INTEGER NOT NULL,
    borrowableItemID INTEGER NOT NULL,
    employeeID INTEGER,
    borrowDateTime TEXT NOT NULL,
    dueDateTime TEXT NOT NULL,
    returnDateTime TEXT,

    CHECK (
        strftime(
            '%Y-%m-%d %H:%M',
            julianday(borrowDateTime)
        ) = borrowDateTime
    ),

    CHECK (
        strftime(
            '%Y-%m-%d %H:%M',
            julianday(dueDateTime)
        ) = dueDateTime
    ),

    CHECK (
        returnDateTime IS NULL
        OR strftime(
            '%Y-%m-%d %H:%M',
            julianday(returnDateTime)
        ) = returnDateTime
    ),

    CHECK (
        julianday(dueDateTime)
        > julianday(borrowDateTime)
    ),

    CHECK (
        returnDateTime IS NULL
        OR julianday(returnDateTime)
           >= julianday(borrowDateTime)
    ),

    FOREIGN KEY (memberID)
        REFERENCES Member(memberID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (borrowableItemID)
        REFERENCES BorrowableItem(borrowableItemID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (employeeID)
        REFERENCES Employee(employeeID)
        ON UPDATE CASCADE
        ON DELETE SET NULL
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Fine table

In [12]:
%%sql

CREATE TABLE Fine (
    fineID INTEGER PRIMARY KEY,
    loanID INTEGER NOT NULL UNIQUE,

    amount REAL NOT NULL
        CHECK (amount >= 0),

    issueDate TEXT NOT NULL,

    paymentStatus TEXT NOT NULL
        CHECK (
            paymentStatus IN (
                'Unpaid',
                'Paid',
                'Waived'
            )
        ),

    paymentDate TEXT,
    description TEXT,

    CHECK (
        issueDate GLOB
            '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
        AND date(julianday(issueDate)) = issueDate
    ),

    CHECK (
        paymentDate IS NULL
        OR (
            paymentDate GLOB
                '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
            AND date(julianday(paymentDate)) = paymentDate
        )
    ),

    CHECK (
        (
            paymentStatus = 'Paid'
            AND paymentDate IS NOT NULL
        )
        OR (
            paymentStatus IN ('Unpaid', 'Waived')
            AND paymentDate IS NULL
        )
    ),

    FOREIGN KEY (loanID)
        REFERENCES Loan(loanID)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Room table

In [13]:
%%sql
CREATE TABLE Room (
    roomID INTEGER PRIMARY KEY,
    roomName TEXT NOT NULL UNIQUE,
    floor INTEGER,
    roomType TEXT NOT NULL
        CHECK (roomType IN (
            'Meeting Room',
            'Activity Room',
            'Exhibition Room',
            'Gaming Room',
            'Study Room'
        )),
    maximumCapacity INTEGER NOT NULL CHECK (maximumCapacity > 0)
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Event table

In [14]:
%%sql

CREATE TABLE Event (
    eventID INTEGER PRIMARY KEY,
    eventName TEXT NOT NULL,

    eventType TEXT NOT NULL
        CHECK (
            eventType IN (
                'Book Club',
                'Author Talk',
                'Book Workshop',
                'Art Show',
                'Film Screening',
                'Cultural Festival',
                'Gaming Event',
                'Group Meeting'
            )
        ),

    description TEXT,
    eventDate TEXT NOT NULL,
    startTime TEXT NOT NULL,
    endTime TEXT NOT NULL,
    roomID INTEGER NOT NULL,

    eventCapacity INTEGER NOT NULL
        CHECK (eventCapacity > 0),

    eventStatus TEXT NOT NULL
        CHECK (
            eventStatus IN (
                'Open',
                'Closed',
                'Cancelled'
            )
        ),

    CHECK (
        eventDate GLOB
            '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
        AND date(julianday(eventDate)) = eventDate
    ),

    CHECK (
        strftime('%H:%M', time(startTime)) = startTime
    ),

    CHECK (
        strftime('%H:%M', time(endTime)) = endTime
    ),

    CHECK (
        time(endTime) > time(startTime)
    ),

    FOREIGN KEY (roomID)
        REFERENCES Room(roomID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the OrganizedBy relationship table

In [15]:
%%sql
CREATE TABLE OrganizedBy (
    employeeID INTEGER NOT NULL,
    eventID INTEGER NOT NULL,
    PRIMARY KEY (employeeID, eventID),
    FOREIGN KEY (employeeID) REFERENCES Employee(employeeID)
        ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (eventID) REFERENCES Event(eventID)
        ON UPDATE CASCADE ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Audience table

In [16]:
%%sql
CREATE TABLE Audience (
    audienceID INTEGER PRIMARY KEY,
    audienceName TEXT NOT NULL UNIQUE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the EventAudience relationship table

In [17]:
%%sql
CREATE TABLE EventAudience (
    eventID INTEGER NOT NULL,
    audienceID INTEGER NOT NULL,
    PRIMARY KEY (eventID, audienceID),
    FOREIGN KEY (eventID) REFERENCES Event(eventID)
        ON UPDATE CASCADE ON DELETE CASCADE,
    FOREIGN KEY (audienceID) REFERENCES Audience(audienceID)
        ON UPDATE CASCADE ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the EventInterest relationship table

In [18]:
%%sql
CREATE TABLE EventInterest (
    eventID INTEGER NOT NULL,
    interestID INTEGER NOT NULL,
    PRIMARY KEY (eventID, interestID),
    FOREIGN KEY (eventID) REFERENCES Event(eventID)
        ON UPDATE CASCADE ON DELETE CASCADE,
    FOREIGN KEY (interestID) REFERENCES Interest(interestID)
        ON UPDATE CASCADE ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the PersonInterest relationship table

In [19]:
%%sql
CREATE TABLE PersonInterest (
    personID INTEGER NOT NULL,
    interestID INTEGER NOT NULL,
    PRIMARY KEY (personID, interestID),
    FOREIGN KEY (personID) REFERENCES Person(personID)
        ON UPDATE CASCADE ON DELETE CASCADE,
    FOREIGN KEY (interestID) REFERENCES Interest(interestID)
        ON UPDATE CASCADE ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the EventRegistration table

In [20]:
%%sql

CREATE TABLE EventRegistration (
    registrationID INTEGER PRIMARY KEY,
    personID INTEGER NOT NULL,
    eventID INTEGER NOT NULL,
    registrationDate TEXT NOT NULL,

    registrationStatus TEXT NOT NULL
        CHECK (
            registrationStatus IN (
                'Registered',
                'Cancelled',
                'Attended',
                'No-show'
            )
        ),

    UNIQUE (personID, eventID),

    CHECK (
        registrationDate GLOB
            '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
        AND date(julianday(registrationDate)) = registrationDate
    ),

    FOREIGN KEY (personID)
        REFERENCES Person(personID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (eventID)
        REFERENCES Event(eventID)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the VolunteerAssignment table

In [21]:
%%sql
CREATE TABLE VolunteerAssignment (
    assignmentID INTEGER PRIMARY KEY,
    role TEXT NOT NULL
        CHECK (role IN (
            'Registration Assistant',
            'Room Setup',
            'Event Guide',
            'Cleanup Assistant'
        )),
    personID INTEGER NOT NULL,
    eventID INTEGER NOT NULL,
    status TEXT NOT NULL
        CHECK (status IN ('Pending', 'Approved', 'Rejected', 'Completed')),
    numberOfHours REAL NOT NULL DEFAULT 0 CHECK (numberOfHours >= 0),
    UNIQUE (personID, eventID, role),
    FOREIGN KEY (personID) REFERENCES Person(personID)
        ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (eventID) REFERENCES Event(eventID)
        ON UPDATE CASCADE ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the Donation table

In [22]:
%%sql

CREATE TABLE Donation (
    donationID INTEGER PRIMARY KEY,
    donorPersonID INTEGER NOT NULL,
    reviewEmployeeID INTEGER,
    donationDate TEXT NOT NULL,

    donationStatus TEXT NOT NULL
        CHECK (
            donationStatus IN (
                'Pending',
                'Approved',
                'Rejected'
            )
        ),

    CHECK (
        donationDate GLOB
            '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
        AND date(julianday(donationDate)) = donationDate
    ),

    FOREIGN KEY (donorPersonID)
        REFERENCES Person(personID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (reviewEmployeeID)
        REFERENCES Employee(employeeID)
        ON UPDATE CASCADE
        ON DELETE SET NULL
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the DonatedItems table

In [23]:
%%sql
CREATE TABLE DonatedItems (
    donatedItemID INTEGER PRIMARY KEY,
    donationID INTEGER NOT NULL,
    title TEXT NOT NULL,
    itemType TEXT NOT NULL,
    author TEXT,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    itemCondition TEXT
        CHECK (
            itemCondition IS NULL
            OR itemCondition IN ('New', 'Good', 'Fair', 'Damaged')
        ),
    approvalStatus TEXT NOT NULL
        CHECK (approvalStatus IN ('Pending', 'Approved', 'Rejected')),
    FOREIGN KEY (donationID) REFERENCES Donation(donationID)
        ON UPDATE CASCADE ON DELETE CASCADE
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the FutureAcquisitions table

In [24]:
%%sql

CREATE TABLE FutureAcquisitions (
    acquisitionID INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    itemType TEXT NOT NULL,
    author TEXT,

    proposedQuantity INTEGER NOT NULL
        CHECK (proposedQuantity > 0),

    estimatedCost REAL
        CHECK (
            estimatedCost IS NULL
            OR estimatedCost >= 0
        ),

    requestDate TEXT NOT NULL,

    requestStatus TEXT NOT NULL
        CHECK (
            requestStatus IN (
                'Proposed',
                'Under Review',
                'Approved',
                'Rejected',
                'Acquired'
            )
        ),

    personID INTEGER NOT NULL,
    employeeID INTEGER,

    CHECK (
        requestDate GLOB
            '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
        AND date(julianday(requestDate)) = requestDate
    ),

    FOREIGN KEY (personID)
        REFERENCES Person(personID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (employeeID)
        REFERENCES Employee(employeeID)
        ON UPDATE CASCADE
        ON DELETE SET NULL
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the application authentication table

In [25]:
%%sql

CREATE TABLE Auth (
    accountID INTEGER PRIMARY KEY,

    personID INTEGER NOT NULL UNIQUE,

    passwordHash TEXT NOT NULL
        CHECK (length(passwordHash) > 0),

    accountRole TEXT NOT NULL
        CHECK (
            accountRole IN (
                'User',
                'Employee'
            )
        ),

    createdAt TEXT NOT NULL,

    CHECK (
        strftime(
            '%Y-%m-%d %H:%M',
            julianday(createdAt)
        ) = createdAt
    ),

    FOREIGN KEY (personID)
        REFERENCES Person(personID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the application help-request table

In [26]:
%%sql

CREATE TABLE HelpRequest (
    helpRequestID INTEGER PRIMARY KEY,
    personID INTEGER NOT NULL,
    assignedEmployeeID INTEGER,

    issue TEXT NOT NULL
        CHECK (length(trim(issue)) > 0),

    messages TEXT NOT NULL
        CHECK (length(trim(messages)) > 0),

    requestStatus TEXT NOT NULL
        CHECK (
            requestStatus IN (
                'Open',
                'In Progress',
                'Resolved'
            )
        ),

    response TEXT,
    createdAt TEXT NOT NULL,
    resolvedAt TEXT,

    CHECK (
        strftime(
            '%Y-%m-%d %H:%M',
            julianday(createdAt)
        ) = createdAt
    ),

    CHECK (
        resolvedAt IS NULL
        OR strftime(
            '%Y-%m-%d %H:%M',
            julianday(resolvedAt)
        ) = resolvedAt
    ),

    CHECK (
        (
            requestStatus = 'Open'
            AND assignedEmployeeID IS NULL
            AND response IS NULL
            AND resolvedAt IS NULL
        )
        OR
        (
            requestStatus = 'In Progress'
            AND assignedEmployeeID IS NOT NULL
            AND response IS NOT NULL
            AND length(trim(response)) > 0
            AND resolvedAt IS NULL
        )
        OR
        (
            requestStatus = 'Resolved'
            AND assignedEmployeeID IS NOT NULL
            AND response IS NOT NULL
            AND length(trim(response)) > 0
            AND resolvedAt IS NOT NULL
        )
    ),

    FOREIGN KEY (personID)
        REFERENCES Person(personID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (assignedEmployeeID)
        REFERENCES Employee(employeeID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

Running query in 'sqlite:///library.db'

++
||
++
++

### Creating the application collection-import table

In [27]:
%%sql

CREATE TABLE AddCollection (
    addCollectionID INTEGER PRIMARY KEY,

    sourceType TEXT NOT NULL
        CHECK (
            sourceType IN (
                'Donation',
                'Acquisition'
            )
        ),

    sourceID INTEGER NOT NULL,
    libraryItemID INTEGER NOT NULL UNIQUE,
    processEmployeeID INTEGER NOT NULL,
    processAt TEXT NOT NULL,

    UNIQUE (sourceType, sourceID),

    CHECK (
        strftime(
            '%Y-%m-%d %H:%M',
            julianday(processAt)
        ) = processAt
    ),

    FOREIGN KEY (libraryItemID)
        REFERENCES LibraryItem(itemID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (processEmployeeID)
        REFERENCES Employee(employeeID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);

Running query in 'sqlite:///library.db'

++
||
++
++

## 4. Add indexes and triggers for integrity

The following cell adds the constraints that cannot be represented by simple column-level checks, including active-loan limits, borrowing eligibility, fine limits, room scheduling, event capacity, and authorized employee roles.

### 4.1 Active-loan index and fine calculation

In [28]:
%%sql

CREATE UNIQUE INDEX one_active_loan_per_item
ON Loan(borrowableItemID)
WHERE returnDateTime IS NULL;


CREATE VIEW ExpectedFine AS
WITH OverdueLoan AS (
    SELECT
        l.loanID,
        li.itemType,

        ROUND(
            julianday(l.returnDateTime)
            - julianday(l.dueDateTime),
            6
        ) AS overdueDays,

        ROUND(
            (
                julianday(l.returnDateTime)
                - julianday(l.dueDateTime)
            ) * 24.0,
            6
        ) AS overdueHours

    FROM Loan l

    JOIN BorrowableItem bi
        ON bi.borrowableItemID = l.borrowableItemID

    JOIN LibraryItem li
        ON li.itemID = bi.itemID

    WHERE l.returnDateTime IS NOT NULL
      AND julianday(l.returnDateTime)
          > julianday(l.dueDateTime)
),

FineUnits AS (
    SELECT
        loanID,
        itemType,

        CAST(overdueDays AS INTEGER)
        + CASE
            WHEN overdueDays
                 > CAST(overdueDays AS INTEGER)
            THEN 1
            ELSE 0
          END AS overdueDayUnits,

        CAST(overdueHours AS INTEGER)
        + CASE
            WHEN overdueHours
                 > CAST(overdueHours AS INTEGER)
            THEN 1
            ELSE 0
          END AS overdueHourUnits

    FROM OverdueLoan
)

SELECT
    loanID,

    CASE
        WHEN itemType IN (
            'Print Book',
            'Magazine',
            'Scientific Journal',
            'Record'
        )
        THEN MIN(
            30.0,
            overdueDayUnits * 0.50
        )

        WHEN itemType = 'Laptop'
        THEN MIN(
            30.0,
            overdueHourUnits * 2.00
        )

        WHEN itemType IN (
            'Charger',
            'Marker',
            'Eraser'
        )
        THEN MIN(
            10.0,
            overdueHourUnits * 0.50
        )
    END AS expectedAmount

FROM FineUnits;

Running query in 'sqlite:///library.db'

++
||
++
++

------

### 4.2 Borrowable-item integrity triggers

In [29]:
%%sql

CREATE TRIGGER validate_borrowable_item_before_insert
BEFORE INSERT ON BorrowableItem
BEGIN
    SELECT CASE
        WHEN (
            SELECT itemType
            FROM LibraryItem
            WHERE itemID = NEW.itemID
        ) = 'Online Book'
        THEN RAISE(
            ABORT,
            'Online books cannot have borrowable item records'
        )
    END;

    SELECT CASE
        WHEN NEW.itemStatus = 'Borrowed'
        THEN RAISE(
            ABORT,
            'New item cannot be Borrowed without active loan'
        )
    END;
END;


CREATE TRIGGER validate_borrowable_status_before_update
BEFORE UPDATE OF itemStatus ON BorrowableItem
WHEN NEW.itemStatus IS NOT OLD.itemStatus
BEGIN
    SELECT CASE
        WHEN NEW.itemStatus = 'Borrowed'
             AND NOT EXISTS (
                 SELECT 1
                 FROM Loan
                 WHERE borrowableItemID
                       = NEW.borrowableItemID
                   AND returnDateTime IS NULL
             )
        THEN RAISE(
            ABORT,
            'Borrowed status requires active loan'
        )
    END;

    SELECT CASE
        WHEN NEW.itemStatus <> 'Borrowed'
             AND EXISTS (
                 SELECT 1
                 FROM Loan
                 WHERE borrowableItemID
                       = NEW.borrowableItemID
                   AND returnDateTime IS NULL
             )
        THEN RAISE(
            ABORT,
            'Item with active loan must remain Borrowed'
        )
    END;
END;

Running query in 'sqlite:///library.db'

++
||
++
++

-------------

### 4.3 Loan and return triggers

In [30]:
%%sql

CREATE TRIGGER validate_loan_before_insert
BEFORE INSERT ON Loan
BEGIN
    SELECT CASE
        WHEN NEW.returnDateTime IS NOT NULL
        THEN RAISE(
            ABORT,
            'New loan must be open and process its return using UPDATE'
        )
    END;

    SELECT CASE
        WHEN (
            SELECT membershipStatus
            FROM Member
            WHERE memberID = NEW.memberID
        ) <> 'Active'
        THEN RAISE(
            ABORT,
            'Only active members can borrow items'
        )
    END;

    SELECT CASE
        WHEN NEW.employeeID IS NOT NULL
             AND NOT EXISTS (
                 SELECT 1
                 FROM Employee
                 WHERE employeeID = NEW.employeeID
                   AND employeeStatus = 'Active'
                   AND position IN (
                       'Librarian',
                       'Library Assistant',
                       'Manager'
                   )
             )
        THEN RAISE(
            ABORT,
            'Loan employee is not authorized'
        )
    END;

    SELECT CASE
        WHEN (
            SELECT itemStatus
            FROM BorrowableItem
            WHERE borrowableItemID
                  = NEW.borrowableItemID
        ) <> 'Available'
        THEN RAISE(
            ABORT,
            'Borrowable item is not available'
        )
    END;

    SELECT CASE
        WHEN (
            SELECT COUNT(*)
            FROM Loan
            WHERE memberID = NEW.memberID
              AND returnDateTime IS NULL
        ) >= 5
        THEN RAISE(
            ABORT,
            'Member already has five active loans'
        )
    END;

    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM Fine f
            JOIN Loan l
                ON l.loanID = f.loanID
            WHERE l.memberID = NEW.memberID
              AND f.paymentStatus = 'Unpaid'
        )
        THEN RAISE(
            ABORT,
            'Member has unpaid fine'
        )
    END;

    SELECT CASE
        WHEN (
            SELECT li.itemType
            FROM BorrowableItem bi
            JOIN LibraryItem li
                ON li.itemID = bi.itemID
            WHERE bi.borrowableItemID
                  = NEW.borrowableItemID
        ) IN (
            'Print Book',
            'Magazine',
            'Scientific Journal',
            'Record'
        )
        AND ABS(
            (
                julianday(NEW.dueDateTime)
                - julianday(NEW.borrowDateTime)
            ) - 21.0
        ) > 0.000001
        THEN RAISE(
            ABORT,
            'Standard materials use 21-day loan'
        )
    END;

    SELECT CASE
        WHEN (
            SELECT li.itemType
            FROM BorrowableItem bi
            JOIN LibraryItem li
                ON li.itemID = bi.itemID
            WHERE bi.borrowableItemID
                  = NEW.borrowableItemID
        ) = 'Laptop'
        AND (
            julianday(NEW.dueDateTime)
            - julianday(NEW.borrowDateTime)
        ) > 1.000001
        THEN RAISE(
            ABORT,
            'Laptop loans have maximum duration of 24 hours'
        )
    END;

    SELECT CASE
        WHEN (
            SELECT li.itemType
            FROM BorrowableItem bi
            JOIN LibraryItem li
                ON li.itemID = bi.itemID
            WHERE bi.borrowableItemID
                  = NEW.borrowableItemID
        ) IN (
            'Charger',
            'Marker',
            'Eraser'
        )
        AND (
            julianday(NEW.dueDateTime)
            - julianday(NEW.borrowDateTime)
        ) > (
            4.0 / 24.0 + 0.000001
        )
        THEN RAISE(
            ABORT,
            'Small equipment loans have maximum duration of 4 hours'
        )
    END;
END;


CREATE TRIGGER prevent_loan_core_update
BEFORE UPDATE OF
    memberID,
    borrowableItemID,
    employeeID,
    borrowDateTime,
    dueDateTime
ON Loan
BEGIN
    SELECT RAISE(
        ABORT,
        'Loan member, item, employee and borrowing dates cannot be changed'
    );
END;


CREATE TRIGGER validate_loan_return_before_update
BEFORE UPDATE OF returnDateTime ON Loan
BEGIN
    SELECT CASE
        WHEN OLD.returnDateTime IS NOT NULL
             AND NEW.returnDateTime IS NOT OLD.returnDateTime
        THEN RAISE(
            ABORT,
            'Completed loan return cannot be changed or reopened'
        )
    END;

    SELECT CASE
        WHEN OLD.returnDateTime IS NULL
             AND NEW.returnDateTime IS NULL
        THEN RAISE(
            ABORT,
            'Return date must be provided when returning item'
        )
    END;

    SELECT CASE
        WHEN NEW.returnDateTime IS NOT NULL
             AND julianday(NEW.returnDateTime)
                 < julianday(NEW.borrowDateTime)
        THEN RAISE(
            ABORT,
            'Return date cannot be before borrow date'
        )
    END;
END;


CREATE TRIGGER update_item_status_after_loan_insert
AFTER INSERT ON Loan
BEGIN
    UPDATE BorrowableItem
    SET itemStatus = 'Borrowed'
    WHERE borrowableItemID = NEW.borrowableItemID;
END;


CREATE TRIGGER update_item_status_after_return
AFTER UPDATE OF returnDateTime ON Loan
WHEN OLD.returnDateTime IS NULL
 AND NEW.returnDateTime IS NOT NULL
BEGIN
    UPDATE BorrowableItem
    SET itemStatus =
        CASE
            WHEN itemCondition = 'Damaged'
            THEN 'Maintenance'
            ELSE 'Available'
        END
    WHERE borrowableItemID = NEW.borrowableItemID;
END;


CREATE TRIGGER prevent_loan_delete
BEFORE DELETE ON Loan
BEGIN
    SELECT RAISE(
        ABORT,
        'Loan history cannot be deleted'
    );
END;

Running query in 'sqlite:///library.db'

++
||
++
++

-------

### 4.4 Fine triggers

In [31]:
%%sql

CREATE TRIGGER validate_fine_before_insert
BEFORE INSERT ON Fine
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM ExpectedFine
            WHERE loanID = NEW.loanID
        )
        THEN RAISE(
            ABORT,
            'Fine requires returned overdue loan'
        )
    END;

    SELECT CASE
        WHEN ABS(
            NEW.amount - (
                SELECT expectedAmount
                FROM ExpectedFine
                WHERE loanID = NEW.loanID
            )
        ) > 0.001
        THEN RAISE(
            ABORT,
            'Fine amount does not match required rate'
        )
    END;

    SELECT CASE
        WHEN NEW.issueDate <> (
            SELECT date(returnDateTime)
            FROM Loan
            WHERE loanID = NEW.loanID
        )
        THEN RAISE(
            ABORT,
            'Fine issue date must be the return date'
        )
    END;

    SELECT CASE
        WHEN NEW.paymentStatus = 'Paid'
             AND NEW.paymentDate IS NULL
        THEN RAISE(
            ABORT,
            'Paid fines require payment date'
        )
    END;

    SELECT CASE
        WHEN NEW.paymentStatus IN (
            'Unpaid',
            'Waived'
        )
        AND NEW.paymentDate IS NOT NULL
        THEN RAISE(
            ABORT,
            'Unpaid and waived fines cannot have payment date'
        )
    END;
END;


CREATE TRIGGER validate_fine_before_update
BEFORE UPDATE ON Fine
BEGIN
    SELECT CASE
        WHEN NEW.loanID IS NOT OLD.loanID
             OR NEW.amount IS NOT OLD.amount
             OR NEW.issueDate IS NOT OLD.issueDate
        THEN RAISE(
            ABORT,
            'Fine loan, amount and issue date cannot be changed'
        )
    END;

    SELECT CASE
        WHEN NEW.paymentStatus = 'Paid'
             AND NEW.paymentDate IS NULL
        THEN RAISE(
            ABORT,
            'Paid fines require payment date'
        )
    END;

    SELECT CASE
        WHEN NEW.paymentStatus IN (
            'Unpaid',
            'Waived'
        )
        AND NEW.paymentDate IS NOT NULL
        THEN RAISE(
            ABORT,
            'Unpaid and waived fines cannot have payment date'
        )
    END;

    SELECT CASE
        WHEN OLD.paymentStatus = 'Paid'
             AND NEW.paymentStatus <> 'Paid'
        THEN RAISE(
            ABORT,
            'Paid fine cannot become unpaid or waived'
        )
    END;

    SELECT CASE
        WHEN OLD.paymentStatus = 'Waived'
             AND NEW.paymentStatus <> 'Waived'
        THEN RAISE(
            ABORT,
            'Waived fine cannot be reactivated'
        )
    END;
END;


CREATE TRIGGER create_fine_after_late_return
AFTER UPDATE OF returnDateTime ON Loan
WHEN OLD.returnDateTime IS NULL
 AND NEW.returnDateTime IS NOT NULL
 AND julianday(NEW.returnDateTime)
     > julianday(NEW.dueDateTime)
BEGIN
    INSERT INTO Fine (
        loanID,
        amount,
        issueDate,
        paymentStatus,
        paymentDate,
        description
    )
    SELECT
        NEW.loanID,
        expectedAmount,
        date(NEW.returnDateTime),
        'Unpaid',
        NULL,
        'Automatically calculated overdue fine'
    FROM ExpectedFine
    WHERE loanID = NEW.loanID;
END;

Running query in 'sqlite:///library.db'

++
||
++
++

-------

### 4.5 Room and event triggers

In [32]:
%%sql

CREATE TRIGGER validate_room_capacity_before_update
BEFORE UPDATE OF maximumCapacity ON Room
BEGIN
    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM Event
            WHERE roomID = NEW.roomID
              AND eventCapacity > NEW.maximumCapacity
        )
        THEN RAISE(
            ABORT,
            'Room capacity cannot be below assigned event capacity'
        )
    END;
END;


CREATE TRIGGER validate_event_before_insert
BEFORE INSERT ON Event
BEGIN
    SELECT CASE
        WHEN NEW.eventCapacity > (
            SELECT maximumCapacity
            FROM Room
            WHERE roomID = NEW.roomID
        )
        THEN RAISE(
            ABORT,
            'Event capacity exceeds room capacity'
        )
    END;

    SELECT CASE
        WHEN time(NEW.endTime) <= time(NEW.startTime)
        THEN RAISE(
            ABORT,
            'Event end time must be later than start time'
        )
    END;

    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM Event e
            WHERE e.roomID = NEW.roomID
              AND e.eventDate = NEW.eventDate
              AND time(NEW.startTime) < time(e.endTime)
              AND time(NEW.endTime) > time(e.startTime)
        )
        THEN RAISE(
            ABORT,
            'Another event overlaps in the same room'
        )
    END;

    SELECT CASE
        WHEN NEW.eventStatus = 'Open'
        THEN RAISE(
            ABORT,
            'Insert event as Closed, assign organizer, then open it'
        )
    END;
END;


CREATE TRIGGER validate_event_before_update
BEFORE UPDATE ON Event
BEGIN
    SELECT CASE
        WHEN NEW.eventCapacity > (
            SELECT maximumCapacity
            FROM Room
            WHERE roomID = NEW.roomID
        )
        THEN RAISE(
            ABORT,
            'Event capacity exceeds room capacity'
        )
    END;

    SELECT CASE
        WHEN time(NEW.endTime) <= time(NEW.startTime)
        THEN RAISE(
            ABORT,
            'Event end time must be later than start time'
        )
    END;

    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM Event e
            WHERE e.roomID = NEW.roomID
              AND e.eventDate = NEW.eventDate
              AND e.eventID <> NEW.eventID
              AND time(NEW.startTime) < time(e.endTime)
              AND time(NEW.endTime) > time(e.startTime)
        )
        THEN RAISE(
            ABORT,
            'Another event overlaps in the same room'
        )
    END;

    SELECT CASE
        WHEN NEW.eventCapacity < (
            SELECT COUNT(*)
            FROM EventRegistration
            WHERE eventID = NEW.eventID
              AND registrationStatus <> 'Cancelled'
        )
        THEN RAISE(
            ABORT,
            'Event capacity cannot be below current registrations'
        )
    END;

    SELECT CASE
        WHEN NEW.eventStatus = 'Open'
             AND NOT EXISTS (
                 SELECT 1
                 FROM OrganizedBy
                 WHERE eventID = NEW.eventID
             )
        THEN RAISE(
            ABORT,
            'Event requires authorized organizer before opening'
        )
    END;
END;

Running query in 'sqlite:///library.db'

++
||
++
++

------

### 4.6 Event-organizer triggers

An event is first created as Closed so its OrganizedBy rows can be added. It can be changed to Open only after it has at least one active authorized organizer. Closed is used as setup state because Event must exist before OrganizedBy can reference it.

In [33]:
%%sql

CREATE TRIGGER validate_organizer_before_insert
BEFORE INSERT ON OrganizedBy
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.employeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Event Coordinator',
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Employee is not authorized to organize events'
        )
    END;
END;


CREATE TRIGGER validate_organizer_before_update
BEFORE UPDATE ON OrganizedBy
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.employeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Event Coordinator',
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Employee is not authorized to organize events'
        )
    END;

    SELECT CASE
        WHEN NEW.eventID <> OLD.eventID
             AND (
                 SELECT eventStatus
                 FROM Event
                 WHERE eventID = OLD.eventID
             ) = 'Open'
             AND (
                 SELECT COUNT(*)
                 FROM OrganizedBy
                 WHERE eventID = OLD.eventID
             ) <= 1
        THEN RAISE(
            ABORT,
            'Open event must retain at least one organizer'
        )
    END;
END;


CREATE TRIGGER validate_organizer_before_delete
BEFORE DELETE ON OrganizedBy
BEGIN
    SELECT CASE
        WHEN (
            SELECT eventStatus
            FROM Event
            WHERE eventID = OLD.eventID
        ) = 'Open'
        AND (
            SELECT COUNT(*)
            FROM OrganizedBy
            WHERE eventID = OLD.eventID
        ) <= 1
        THEN RAISE(
            ABORT,
            'Last organizer of open event cannot be removed'
        )
    END;
END;

Running query in 'sqlite:///library.db'

++
||
++
++

------

### 4.7 Event-registration triggers

In [34]:
%%sql

CREATE TRIGGER validate_registration_before_insert
BEFORE INSERT ON EventRegistration
BEGIN
    SELECT CASE
        WHEN (
            SELECT eventStatus
            FROM Event
            WHERE eventID = NEW.eventID
        ) <> 'Open'
        THEN RAISE(
            ABORT,
            'Registration is allowed only for open events'
        )
    END;

    SELECT CASE
        WHEN NEW.registrationStatus <> 'Cancelled'
             AND (
                 SELECT COUNT(*)
                 FROM EventRegistration
                 WHERE eventID = NEW.eventID
                   AND registrationStatus <> 'Cancelled'
             ) >= (
                 SELECT eventCapacity
                 FROM Event
                 WHERE eventID = NEW.eventID
             )
        THEN RAISE(
            ABORT,
            'Event capacity has been reached'
        )
    END;

    SELECT CASE
        WHEN EXISTS (
            SELECT 1
            FROM Member m
            JOIN Loan l
                ON l.memberID = m.memberID
            JOIN Fine f
                ON f.loanID = l.loanID
            WHERE m.personID = NEW.personID
              AND f.paymentStatus = 'Unpaid'
        )
        THEN RAISE(
            ABORT,
            'Person is member with unpaid fine'
        )
    END;
END;


CREATE TRIGGER validate_registration_before_update
BEFORE UPDATE ON EventRegistration
BEGIN
    SELECT CASE
        WHEN NEW.personID <> OLD.personID
             OR NEW.eventID <> OLD.eventID
        THEN RAISE(
            ABORT,
            'Registration person and event cannot be changed'
        )
    END;

    SELECT CASE
        WHEN OLD.registrationStatus = 'Cancelled'
             AND NEW.registrationStatus <> 'Cancelled'
             AND (
                 SELECT eventStatus
                 FROM Event
                 WHERE eventID = NEW.eventID
             ) <> 'Open'
        THEN RAISE(
            ABORT,
            'Cancelled registration can only be restored for open event'
        )
    END;

    SELECT CASE
        WHEN OLD.registrationStatus = 'Cancelled'
             AND NEW.registrationStatus <> 'Cancelled'
             AND (
                 SELECT COUNT(*)
                 FROM EventRegistration
                 WHERE eventID = NEW.eventID
                   AND registrationStatus <> 'Cancelled'
                   AND registrationID <> NEW.registrationID
             ) >= (
                 SELECT eventCapacity
                 FROM Event
                 WHERE eventID = NEW.eventID
             )
        THEN RAISE(
            ABORT,
            'Event capacity has been reached'
        )
    END;

    SELECT CASE
        WHEN OLD.registrationStatus = 'Cancelled'
             AND NEW.registrationStatus <> 'Cancelled'
             AND EXISTS (
                 SELECT 1
                 FROM Member m
                 JOIN Loan l
                     ON l.memberID = m.memberID
                 JOIN Fine f
                     ON f.loanID = l.loanID
                 WHERE m.personID = NEW.personID
                   AND f.paymentStatus = 'Unpaid'
             )
        THEN RAISE(
            ABORT,
            'Person is member with unpaid fine'
        )
    END;
END;

Running query in 'sqlite:///library.db'

++
||
++
++

-----

### 4.8 Donation integrity triggers

A donation is first created as Pending, then its DonatedItems rows are added. It cannot become Approved or Rejected without at least one item. Pending is used as setup state because Donation must exist before DonatedItems can reference it.

In [35]:
%%sql

CREATE TRIGGER validate_donation_before_insert
BEFORE INSERT ON Donation
BEGIN
    SELECT CASE
        WHEN NEW.donationStatus <> 'Pending'
        THEN RAISE(
            ABORT,
            'New donation must initially be Pending'
        )
    END;

    SELECT CASE
        WHEN NEW.reviewEmployeeID IS NOT NULL
             AND NOT EXISTS (
                 SELECT 1
                 FROM Employee
                 WHERE employeeID = NEW.reviewEmployeeID
                   AND employeeStatus = 'Active'
                   AND position IN (
                       'Librarian',
                       'Manager'
                   )
             )
        THEN RAISE(
            ABORT,
            'Donation reviewer is not authorized'
        )
    END;
END;


CREATE TRIGGER validate_donation_reviewer_before_update
BEFORE UPDATE OF reviewEmployeeID ON Donation
WHEN NEW.reviewEmployeeID IS NOT NULL
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.reviewEmployeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Donation reviewer is not authorized'
        )
    END;
END;


CREATE TRIGGER validate_donation_status_before_update
BEFORE UPDATE OF donationStatus ON Donation
BEGIN
    SELECT CASE
        WHEN OLD.donationStatus IN (
            'Approved',
            'Rejected'
        )
        AND NEW.donationStatus <> OLD.donationStatus
        THEN RAISE(
            ABORT,
            'Finalized donation status cannot be changed'
        )
    END;

    SELECT CASE
        WHEN NEW.donationStatus IN (
            'Approved',
            'Rejected'
        )
        AND NEW.reviewEmployeeID IS NULL
        THEN RAISE(
            ABORT,
            'Approved or rejected donations require reviewer'
        )
    END;

    SELECT CASE
        WHEN NEW.donationStatus IN (
            'Approved',
            'Rejected'
        )
        AND NOT EXISTS (
            SELECT 1
            FROM DonatedItems
            WHERE donationID = NEW.donationID
        )
        THEN RAISE(
            ABORT,
            'Donation must contain at least one donated item'
        )
    END;

    SELECT CASE
        WHEN NEW.donationStatus = 'Approved'
             AND NOT EXISTS (
                 SELECT 1
                 FROM DonatedItems
                 WHERE donationID = NEW.donationID
                   AND approvalStatus = 'Approved'
             )
        THEN RAISE(
            ABORT,
            'Approved donation requires at least one approved donated item'
        )
    END;

    SELECT CASE
        WHEN NEW.donationStatus IN (
            'Approved',
            'Rejected'
        )
        AND NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.reviewEmployeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Donation reviewer is not authorized'
        )
    END;
END;


CREATE TRIGGER prevent_donated_item_donation_change
BEFORE UPDATE OF donationID ON DonatedItems
BEGIN
    SELECT RAISE(
        ABORT,
        'Donated item cannot be moved to another donation'
    );
END;


CREATE TRIGGER protect_last_approved_item_before_update
BEFORE UPDATE OF approvalStatus ON DonatedItems
WHEN OLD.approvalStatus = 'Approved'
 AND NEW.approvalStatus <> 'Approved'
 AND (
     SELECT donationStatus
     FROM Donation
     WHERE donationID = OLD.donationID
 ) = 'Approved'
 AND NOT EXISTS (
     SELECT 1
     FROM DonatedItems
     WHERE donationID = OLD.donationID
       AND donatedItemID <> OLD.donatedItemID
       AND approvalStatus = 'Approved'
 )
BEGIN
    SELECT RAISE(
        ABORT,
        'Approved donation needs approved item'
    );
END;


CREATE TRIGGER protect_last_approved_item_before_delete
BEFORE DELETE ON DonatedItems
WHEN OLD.approvalStatus = 'Approved'
 AND (
     SELECT donationStatus
     FROM Donation
     WHERE donationID = OLD.donationID
 ) = 'Approved'
 AND NOT EXISTS (
     SELECT 1
     FROM DonatedItems
     WHERE donationID = OLD.donationID
       AND donatedItemID <> OLD.donatedItemID
       AND approvalStatus = 'Approved'
 )
BEGIN
    SELECT RAISE(
        ABORT,
        'Approved donation needs approved item'
    );
END;



CREATE TRIGGER prevent_last_finalized_donated_item_delete
BEFORE DELETE ON DonatedItems
WHEN (
    SELECT donationStatus
    FROM Donation
    WHERE donationID = OLD.donationID
) IN ('Approved', 'Rejected')
AND (
    SELECT COUNT(*)
    FROM DonatedItems
    WHERE donationID = OLD.donationID
) <= 1
BEGIN
    SELECT RAISE(
        ABORT,
        'Finalized donation must retain at least one donated item'
    );
END;

Running query in 'sqlite:///library.db'

++
||
++
++

-----

### 4.9 Acquisition integrity triggers

In [36]:
%%sql

CREATE TRIGGER validate_acquisition_before_insert
BEFORE INSERT ON FutureAcquisitions
BEGIN
    SELECT CASE
        WHEN NEW.requestStatus = 'Acquired'
        THEN RAISE(
            ABORT,
            'Request must be Approved before becoming Acquired'
        )
    END;

    SELECT CASE
        WHEN NEW.employeeID IS NOT NULL
             AND NOT EXISTS (
                 SELECT 1
                 FROM Employee
                 WHERE employeeID = NEW.employeeID
                   AND employeeStatus = 'Active'
                   AND position IN (
                       'Librarian',
                       'Manager'
                   )
             )
        THEN RAISE(
            ABORT,
            'Acquisition reviewer is not authorized'
        )
    END;

    SELECT CASE
        WHEN NEW.requestStatus IN (
            'Approved',
            'Rejected'
        )
        AND NEW.employeeID IS NULL
        THEN RAISE(
            ABORT,
            'Approved or rejected requests require reviewer'
        )
    END;
END;


CREATE TRIGGER validate_acquisition_reviewer_before_update
BEFORE UPDATE OF employeeID ON FutureAcquisitions
WHEN NEW.employeeID IS NOT NULL
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.employeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Acquisition reviewer is not authorized'
        )
    END;
END;

CREATE TRIGGER validate_acquisition_status_before_update
BEFORE UPDATE OF requestStatus ON FutureAcquisitions
BEGIN
    SELECT CASE
        WHEN OLD.requestStatus = 'Acquired'
             AND NEW.requestStatus <> 'Acquired'
        THEN RAISE(
            ABORT,
            'Acquired request cannot be reopened'
        )
    END;

    SELECT CASE
        WHEN NEW.requestStatus IN (
            'Approved',
            'Rejected',
            'Acquired'
        )
        AND NEW.employeeID IS NULL
        THEN RAISE(
            ABORT,
            'Reviewed acquisition statuses require reviewer'
        )
    END;

    SELECT CASE
        WHEN NEW.requestStatus IN (
            'Approved',
            'Rejected',
            'Acquired'
        )
        AND NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.employeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Acquisition reviewer is not authorized'
        )
    END;

    SELECT CASE
        WHEN NEW.requestStatus = 'Acquired'
             AND OLD.requestStatus NOT IN (
                 'Approved',
                 'Acquired'
             )
        THEN RAISE(
            ABORT,
            'Only Approved request can become Acquired'
        )
    END;

    SELECT CASE
        WHEN NEW.requestStatus = 'Acquired'
         AND NOT EXISTS (
             SELECT 1
             FROM AddCollection
             WHERE sourceType = 'Acquisition'
               AND sourceID = NEW.acquisitionID
         )
        THEN RAISE(
            ABORT,
            'Acquired request needs collection record'
        )
    END;
END;

Running query in 'sqlite:///library.db'

++
||
++
++

-----

### 4.10 Collection-source triggers

No collection-source triggers are used here. Donation and acquisition approval are handled by their own status and reviewer triggers.

-----

### 4.11 ADDED - Remaining integrity triggers

The triggers below stop item type changes when copies exist, stop moving a copy to another item, protect organizers of open events, check acquisition requester eligibility, prevent fine deletion, and cancel registrations when an event is cancelled.

In [37]:
%%sql

-- Do not change an item's type after physical copies exist.
-- This also protects old loan and fine calculations.
CREATE TRIGGER prevent_item_type_change_with_copies
BEFORE UPDATE OF itemType ON LibraryItem
WHEN NEW.itemType <> OLD.itemType
 AND EXISTS (
     SELECT 1
     FROM BorrowableItem
     WHERE itemID = NEW.itemID
 )
BEGIN
    SELECT RAISE(
        ABORT,
        'Cannot change type while copies exist'
    );
END;


-- A tracked copy should always belong to the same LibraryItem.
CREATE TRIGGER prevent_borrowable_item_parent_change
BEFORE UPDATE OF itemID ON BorrowableItem
WHEN NEW.itemID <> OLD.itemID
BEGIN
    SELECT RAISE(
        ABORT,
        'Cannot move copy to another item'
    );
END;


-- Do not allow an open event's organizer to become inactive
-- or change to a position that cannot organize events.
CREATE TRIGGER protect_open_event_organizer_employee
BEFORE UPDATE OF position, employeeStatus ON Employee
WHEN EXISTS (
    SELECT 1
    FROM OrganizedBy ob
    JOIN Event e
        ON e.eventID = ob.eventID
    WHERE ob.employeeID = OLD.employeeID
      AND e.eventStatus = 'Open'
)
AND (
    NEW.employeeStatus <> 'Active'
    OR NEW.position NOT IN (
        'Event Coordinator',
        'Librarian',
        'Manager'
    )
)
BEGIN
    SELECT RAISE(
        ABORT,
        'Employee is organizer for open event'
    );
END;


-- Check the organizer's current role and status when opening
-- an event. It is not enough for OrganizedBy to just exist.
CREATE TRIGGER check_organizer_when_event_opens
BEFORE UPDATE OF eventStatus ON Event
WHEN NEW.eventStatus = 'Open'
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM OrganizedBy ob
            JOIN Employee emp
                ON emp.employeeID = ob.employeeID
            WHERE ob.eventID = NEW.eventID
              AND emp.employeeStatus = 'Active'
              AND emp.position IN (
                  'Event Coordinator',
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Open event needs active organizer'
        )
    END;
END;




-- Acquisition requester must be a member, employee or donor.
CREATE TRIGGER validate_acquisition_requester_insert
BEFORE INSERT ON FutureAcquisitions
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Member
            WHERE personID = NEW.personID
        )
        AND NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE personID = NEW.personID
        )
        AND NOT EXISTS (
            SELECT 1
            FROM Donation
            WHERE donorPersonID = NEW.personID
        )
        THEN RAISE(
            ABORT,
            'Requester must be member employee or donor'
        )
    END;
END;


-- Requester cannot later be changed to an unrelated person.
CREATE TRIGGER validate_acquisition_requester_update
BEFORE UPDATE OF personID ON FutureAcquisitions
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Member
            WHERE personID = NEW.personID
        )
        AND NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE personID = NEW.personID
        )
        AND NOT EXISTS (
            SELECT 1
            FROM Donation
            WHERE donorPersonID = NEW.personID
        )
        THEN RAISE(
            ABORT,
            'Requester must be member employee or donor'
        )
    END;
END;


-- Deleting a fine could remove the unpaid-fine restriction.
-- Paid and Waived statuses should be used instead.
CREATE TRIGGER prevent_fine_delete
BEFORE DELETE ON Fine
BEGIN
    SELECT RAISE(
        ABORT,
        'Dont delete fine use Paid or Waived'
    );
END;


-- Existing registrations should be cancelled when the
-- entire event is cancelled.
CREATE TRIGGER cancel_registrations_after_event_cancel
AFTER UPDATE OF eventStatus ON Event
WHEN OLD.eventStatus <> 'Cancelled'
 AND NEW.eventStatus = 'Cancelled'
BEGIN
    UPDATE EventRegistration
    SET registrationStatus = 'Cancelled'
    WHERE eventID = NEW.eventID
      AND registrationStatus = 'Registered';
END;

Running query in 'sqlite:///library.db'

++
||
++
++

### 4.12 Application authentication and support integrity

In [38]:
%%sql

CREATE INDEX idx_help_request_status
ON HelpRequest(requestStatus);

CREATE INDEX idx_help_request_person
ON HelpRequest(personID);

CREATE INDEX idx_add_collection_item
ON AddCollection(libraryItemID);


CREATE TRIGGER validate_employee_auth_before_insert
BEFORE INSERT ON Auth
WHEN NEW.accountRole = 'Employee'
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE personID = NEW.personID
        )
        THEN RAISE(
            ABORT,
            'Employee login requires employee record'
        )
    END;
END;


CREATE TRIGGER validate_employee_auth_before_update
BEFORE UPDATE OF personID, accountRole ON Auth
WHEN NEW.accountRole = 'Employee'
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE personID = NEW.personID
        )
        THEN RAISE(
            ABORT,
            'Employee login requires employee record'
        )
    END;
END;


CREATE TRIGGER validate_help_employee_before_insert
BEFORE INSERT ON HelpRequest
WHEN NEW.assignedEmployeeID IS NOT NULL
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.assignedEmployeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Help request employee is not authorized'
        )
    END;
END;


CREATE TRIGGER validate_help_employee_before_update
BEFORE UPDATE OF assignedEmployeeID ON HelpRequest
WHEN NEW.assignedEmployeeID IS NOT NULL
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.assignedEmployeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Help request employee is not authorized'
        )
    END;
END;


CREATE TRIGGER validate_collection_import_before_insert
BEFORE INSERT ON AddCollection
BEGIN
    SELECT CASE
        WHEN NOT EXISTS (
            SELECT 1
            FROM Employee
            WHERE employeeID = NEW.processEmployeeID
              AND employeeStatus = 'Active'
              AND position IN (
                  'Librarian',
                  'Manager'
              )
        )
        THEN RAISE(
            ABORT,
            'Collection employee is not authorized'
        )
    END;

    SELECT CASE
        WHEN NEW.sourceType = 'Donation'
         AND NOT EXISTS (
             SELECT 1
             FROM DonatedItems di
             JOIN Donation d
                 ON d.donationID = di.donationID
             WHERE di.donatedItemID = NEW.sourceID
               AND di.approvalStatus = 'Approved'
               AND d.donationStatus = 'Approved'
         )
        THEN RAISE(
            ABORT,
            'Donation item must be approved before import'
        )
    END;

    SELECT CASE
        WHEN NEW.sourceType = 'Acquisition'
         AND NOT EXISTS (
             SELECT 1
             FROM FutureAcquisitions
             WHERE acquisitionID = NEW.sourceID
               AND requestStatus = 'Approved'
         )
        THEN RAISE(
            ABORT,
            'Acquisition must be approved before import'
        )
    END;

    SELECT CASE
        WHEN NEW.sourceType = 'Donation'
         AND NOT EXISTS (
             SELECT 1
             FROM DonatedItems di
             JOIN LibraryItem li
                 ON li.itemID = NEW.libraryItemID
             WHERE di.donatedItemID = NEW.sourceID
               AND li.itemTitle = di.title
               AND COALESCE(li.author, '')
                   = COALESCE(di.author, '')
         )
        THEN RAISE(
            ABORT,
            'Library item does not match donation item'
        )
    END;

    SELECT CASE
        WHEN NEW.sourceType = 'Acquisition'
         AND NOT EXISTS (
             SELECT 1
             FROM FutureAcquisitions fa
             JOIN LibraryItem li
                 ON li.itemID = NEW.libraryItemID
             WHERE fa.acquisitionID = NEW.sourceID
               AND li.itemTitle = fa.title
               AND COALESCE(li.author, '')
                   = COALESCE(fa.author, '')
         )
        THEN RAISE(
            ABORT,
            'Library item does not match acquisition'
        )
    END;

    SELECT CASE
        WHEN (
            SELECT itemType
            FROM LibraryItem
            WHERE itemID = NEW.libraryItemID
        ) <> 'Online Book'

        AND (
            SELECT COUNT(*)
            FROM BorrowableItem
            WHERE itemID = NEW.libraryItemID
        ) <> CASE NEW.sourceType
            WHEN 'Donation' THEN (
                SELECT quantity
                FROM DonatedItems
                WHERE donatedItemID = NEW.sourceID
            )

            WHEN 'Acquisition' THEN (
                SELECT proposedQuantity
                FROM FutureAcquisitions
                WHERE acquisitionID = NEW.sourceID
            )
        END

        THEN RAISE(
            ABORT,
            'Physical import needs all borrowable copies'
        )
    END;
END;

CREATE TRIGGER mark_acquisition_acquired_after_import
AFTER INSERT ON AddCollection
WHEN NEW.sourceType = 'Acquisition'
BEGIN
    UPDATE FutureAcquisitions
    SET requestStatus = 'Acquired',
        employeeID = NEW.processEmployeeID
    WHERE acquisitionID = NEW.sourceID;
END;

CREATE TRIGGER prevent_imported_donated_item_update
BEFORE UPDATE ON DonatedItems
WHEN EXISTS (
    SELECT 1
    FROM AddCollection
    WHERE sourceType = 'Donation'
      AND sourceID = OLD.donatedItemID
)
BEGIN
    SELECT RAISE(
        ABORT,
        'Imported donation item cannot be changed'
    );
END;


CREATE TRIGGER prevent_imported_acquisition_details_update
BEFORE UPDATE OF
    title,
    itemType,
    author,
    proposedQuantity
ON FutureAcquisitions
WHEN EXISTS (
    SELECT 1
    FROM AddCollection
    WHERE sourceType = 'Acquisition'
      AND sourceID = OLD.acquisitionID
)
BEGIN
    SELECT RAISE(
        ABORT,
        'Imported acquisition details cannot be changed'
    );
END;


CREATE TRIGGER prevent_imported_library_item_identity_update
BEFORE UPDATE OF
    itemTitle,
    itemType,
    author
ON LibraryItem
WHEN EXISTS (
    SELECT 1
    FROM AddCollection
    WHERE libraryItemID = OLD.itemID
)
BEGIN
    SELECT RAISE(
        ABORT,
        'Imported library item details cannot be changed'
    );
END;


CREATE TRIGGER prevent_imported_donated_item_delete
BEFORE DELETE ON DonatedItems
WHEN EXISTS (
    SELECT 1
    FROM AddCollection
    WHERE sourceType = 'Donation'
      AND sourceID = OLD.donatedItemID
)
BEGIN
    SELECT RAISE(
        ABORT,
        'Imported donation item cannot be deleted'
    );
END;


CREATE TRIGGER prevent_imported_acquisition_delete
BEFORE DELETE ON FutureAcquisitions
WHEN EXISTS (
    SELECT 1
    FROM AddCollection
    WHERE sourceType = 'Acquisition'
      AND sourceID = OLD.acquisitionID
)
BEGIN
    SELECT RAISE(
        ABORT,
        'Imported acquisition cannot be deleted'
    );
END;


CREATE TRIGGER prevent_collection_import_update
BEFORE UPDATE ON AddCollection
BEGIN
    SELECT RAISE(
        ABORT,
        'Collection import record cannot be changed'
    );
END;


CREATE TRIGGER prevent_collection_import_delete
BEFORE DELETE ON AddCollection
BEGIN
    SELECT RAISE(
        ABORT,
        'Collection import history cannot be deleted'
    );
END;

Running query in 'sqlite:///library.db'

++
||
++
++

-----

## 5. Populate every table with realistic data

### Inserting realistic people with varied age groups


In [39]:
%%sql
INSERT INTO Person (personID, firstName, lastName, email, phoneNumber, address, dateOfBirth) VALUES
(1, 'Aisha', 'Khan', 'aisha.khan26@gmail.com', '604-555-0101', '7425 Kingsway, Burnaby, BC', '1994-04-17'),
(2, 'Mateo', 'Rodriguez', 'mateorodriguez@icloud.com', '778-555-0102', '1055 West Georgia Street, Vancouver, BC', '1982-11-03'),
(3, 'Chloe', 'Martin', 'chloe.martin@outlook.com', '604-555-0103', '12888 80 Avenue, Surrey, BC', '2010-07-22'),
(4, 'Arjun', 'Mehta', 'arjunmehta88@gmail.com', '778-555-0104', '6351 No. 3 Road, Richmond, BC', '1964-02-14'),
(5, 'Sofia', 'Rossi', 'sofia.rossi@icloud.com', '604-555-0105', '2929 Barnet Highway, Coquitlam, BC', '2000-09-08'),
(6, 'Noah', 'Williams', 'noahwilliams@proton.me', '778-555-0106', '610 Sixth Street, New Westminster, BC', '1976-05-29'),
(7, 'Yuna', 'Park', 'yuna.park03@gmail.com', '604-555-0107', '145 West 15th Street, North Vancouver, BC', '2017-12-01'),
(8, 'Liam', 'O''Connor', 'liamoconnor@outlook.com', '778-555-0108', '1585 Marine Drive, West Vancouver, BC', '2001-03-19'),
(9, 'Fatima', 'Zahra', 'fatima.zahra@icloud.com', '604-555-0109', '8089 120 Street, Delta, BC', '1955-08-26'),
(10, 'Ethan', 'Chen', 'ethanchen98@gmail.com', '778-555-0110', '20355 64 Avenue, Langley, BC', '2008-06-11'),
(11, 'Grace', 'Liu', 'grace.liu@outlook.com', '604-555-0111', '3015 Murray Street, Port Moody, BC', '1948-10-04'),
(12, 'Omar', 'Hassan', 'omarhassan@icloud.com', '778-555-0112', '2580 Shaughnessy Street, Port Coquitlam, BC', '1973-01-31'),
(13, 'Emily', 'Carter', 'emily.carter@notgooglelibrary.ca', '604-555-0113', '15210 Pacific Avenue, White Rock, BC', '1987-05-16'),
(14, 'Daniel', 'Kim', 'danielkim@notgooglelibrary.ca', '778-555-0114', '22470 Dewdney Trunk Road, Maple Ridge, BC', '1975-09-27'),
(15, 'Priya', 'Patel', 'priya.patel@notgooglelibrary.ca', '604-555-0115', '19070 Lougheed Highway, Pitt Meadows, BC', '1991-02-09'),
(16, 'Lucas', 'Silva', 'lucassilva@notgooglelibrary.ca', '778-555-0116', '4567 Hastings Street, Burnaby, BC', '2000-07-13'),
(17, 'Maya', 'Thompson', 'maya.thompson@notgooglelibrary.ca', '604-555-0117', '2212 Main Street, Vancouver, BC', '1984-12-20'),
(18, 'Gabriel', 'Nakamura', 'gabrielnakamura@notgooglelibrary.ca', '778-555-0118', '10330 152 Street, Surrey, BC', '1971-04-05'),
(19, 'Isabella', 'Brown', 'isabella.brown@notgooglelibrary.ca', '604-555-0119', '8120 Granville Avenue, Richmond, BC', '1993-11-24'),
(20, 'Samuel', 'Adeyemi', 'samueladeyemi@notgooglelibrary.ca', '778-555-0120', '1190 Pinetree Way, Coquitlam, BC', '1980-06-30'),
(21, 'Nora', 'Andersen', 'nora.andersen@notgooglelibrary.ca', '604-555-0121', '721 Front Street, New Westminster, BC', '1958-03-12'),
(22, 'Hugo', 'Laurent', 'hugolaurent@notgooglelibrary.ca', '778-555-0122', '1650 Lonsdale Avenue, North Vancouver, BC', '1998-08-18'),
(23, 'Zainab', 'Ali', 'zainab.ali@gmail.com', '604-555-0123', '1360 Marine Drive, West Vancouver, BC', '2009-01-25'),
(24, 'Jack', 'Murphy', 'jackmurphy@icloud.com', '778-555-0124', '11720 88 Avenue, Delta, BC', '1988-10-15'),
(25, 'Camila', 'Santos', 'camila.santos@outlook.com', '604-555-0125', '19850 Willowbrook Drive, Langley, BC', '2004-04-02'),
(26, 'Wei', 'Zhang', 'weizhang@gmail.com', '778-555-0126', '2800 St Johns Street, Port Moody, BC', '1960-07-07'),
(27, 'Layla', 'Johnson', 'layla.johnson@icloud.com', '604-555-0127', '2331 Kelly Avenue, Port Coquitlam, BC', '2015-09-21'),
(28, 'Benjamin', 'Taylor', 'benjamintaylor@yahoo.com', '778-555-0128', '15165 Thrift Avenue, White Rock, BC', '1942-12-09');


Running query in 'sqlite:///library.db'

28 rows affected.

++
||
++
++

### Inserting members with every membership status


In [40]:
%%sql
INSERT INTO Member (memberID, personID, membershipDate, membershipStatus) VALUES
(1, 1, '2017-01-11', 'Active'),
(2, 2, '2018-02-12', 'Active'),
(3, 3, '2019-03-13', 'Active'),
(4, 4, '2020-04-14', 'Inactive'),
(5, 5, '2021-05-15', 'Suspended'),
(6, 6, '2022-06-16', 'Active'),
(7, 7, '2023-07-17', 'Active'),
(8, 8, '2024-08-18', 'Active'),
(9, 9, '2025-09-19', 'Inactive'),
(10, 10, '2016-10-20', 'Suspended'),
(11, 11, '2017-11-21', 'Active'),
(12, 12, '2018-12-22', 'Active');


Running query in 'sqlite:///library.db'

12 rows affected.

++
||
++
++

### Inserting varied interests


In [41]:
%%sql
INSERT INTO Interest (interestID, interestName) VALUES
(1, 'Literary Fiction'),
(2, 'Science Fiction'),
(3, 'History'),
(4, 'Technology'),
(5, 'Mystery and Crime'),
(6, 'Music'),
(7, 'Art and Design'),
(8, 'Children''s Literature'),
(9, 'Environment'),
(10, 'Film'),
(11, 'Language Learning'),
(12, 'Board Games');


Running query in 'sqlite:///library.db'

12 rows affected.

++
||
++
++

### Inserting employees from young-adult, adult, and senior Person records


In [42]:
%%sql
-- The referenced Person rows include young adults, adults, and a senior.
INSERT INTO Employee (employeeID, personID, position, hireDate, salary, employeeStatus) VALUES
(1, 13, 'Librarian', '2018-04-09', 68500, 'Active'),
(2, 14, 'Manager', '2015-09-21', 84200, 'Active'),
(3, 15, 'Event Coordinator', '2021-01-18', 61200, 'Active'),
(4, 16, 'Library Assistant', '2023-06-12', 47400, 'Active'),
(5, 17, 'Librarian', '2019-11-04', 70250, 'Active'),
(6, 18, 'Manager', '2017-02-27', 81600, 'Active'),
(7, 19, 'Event Coordinator', '2022-08-15', 59600, 'Active'),
(8, 20, 'Library Assistant', '2020-05-25', 48900, 'Inactive'),
(9, 21, 'Technician', '2016-10-03', 63800, 'Active'),
(10, 22, 'Technician', '2024-03-11', 55200, 'On Leave');


Running query in 'sqlite:///library.db'

10 rows affected.

++
||
++
++

### Inserting real books and varied library materials


In [43]:
%%sql
INSERT INTO LibraryItem (itemID, itemTitle, itemType, author, publisher, publicationYear, language, ISBN, onlineAccessURL) VALUES
(1, 'The Hobbit', 'Print Book', 'J. R. R. Tolkien', 'Mariner Books', 2012, 'English', '9780547928227', NULL),
(2, '1984', 'Print Book', 'George Orwell', 'Signet Classics', 1950, 'English', '9780451524935', NULL),
(3, 'Database System Concepts', 'Print Book', 'Silberschatz, Korth, Sudarshan', 'McGraw-Hill', 2019, 'English', '9780078022159', NULL),
(4, 'NASA Spinoff 2024', 'Magazine', 'NASA Spinoff Staff', 'National Aeronautics and Space Administration', 2024, 'English', NULL, 'https://spinoff.nasa.gov/sites/default/files/2024-01/NASA.Spinoff_2024_508.pdf'),
(5, 'Nature: Artificial Intelligence Collection', 'Scientific Journal', 'Nature Editorial Team', 'Springer Nature', 2024, 'English', NULL, NULL),
(6, 'Kind of Blue', 'Record', 'Miles Davis', 'Columbia Records', 1959, 'Instrumental', NULL, NULL),
(7, 'Dell Latitude 5440', 'Laptop', 'Dell Technologies', 'Dell', 2023, 'English', NULL, NULL),
(8, 'Anker 735 USB-C Charger', 'Charger', 'Anker Innovations', 'Anker', 2022, 'English', NULL, NULL),
(9, 'Expo Low Odor Marker Set', 'Marker', 'Newell Brands', 'Expo', 2024, 'English', NULL, NULL),
(10, 'Quartet Whiteboard Eraser', 'Eraser', 'ACCO Brands', 'Quartet', 2023, 'English', NULL, NULL),
(11, 'Pride and Prejudice', 'Online Book', 'Jane Austen', 'Planet eBook', 1813, 'English', NULL, 'https://www.planetebook.com/free-ebooks/pride-and-prejudice.pdf'),
(12, 'Dune', 'Print Book', 'Frank Herbert', 'Ace Books', 1965, 'English', '9780441172719', NULL),
(13, 'The Left Hand of Darkness', 'Print Book', 'Ursula K. Le Guin', 'Ace Books', 1969, 'English', '9780441478125', NULL),
(14, 'Atomic Habits', 'Print Book', 'James Clear', 'Avery', 2018, 'English', '9780735211292', NULL),
(15, 'Blue Train', 'Record', 'John Coltrane', 'Blue Note Records', 1957, 'Instrumental', NULL, NULL),
(16, 'Frankenstein', 'Online Book', 'Mary Shelley', 'Planet eBook', 1818, 'English', NULL, 'https://www.planetebook.com/free-ebooks/frankenstein.pdf'),
(17, 'Little Women', 'Online Book', 'Louisa May Alcott', 'Planet eBook', 1868, 'English', NULL, 'https://www.planetebook.com/free-ebooks/little-women.pdf'),
(18, 'Anne of Green Gables', 'Online Book', 'Lucy Maud Montgomery', 'Planet eBook', 1908, 'English', NULL, 'https://www.planetebook.com/free-ebooks/anne-of-green-gables.pdf'),
(19, 'Alice''s Adventures in Wonderland', 'Online Book', 'Lewis Carroll', 'Planet eBook', 1865, 'English', NULL, 'https://www.planetebook.com/free-ebooks/alices-adventures-in-wonderland.pdf'),
(20, 'Wuthering Heights', 'Online Book', 'Emily Bronte', 'Planet eBook', 1847, 'English', NULL, 'https://www.planetebook.com/free-ebooks/wuthering-heights.pdf'),
(21, 'Penguin Classics eBook Package', 'Online Book', 'Penguin Random House', 'Penguin Random House', 2026, 'English', NULL, 'https://www.penguinrandomhouse.com/penguinclassics/');


Running query in 'sqlite:///library.db'

21 rows affected.

++
||
++
++

### Inserting physical copies with every condition and status


In [44]:
%%sql
INSERT INTO BorrowableItem (borrowableItemID, itemID, barcode, shelfLocation, itemCondition, itemStatus) VALUES
(1, 1, 'BK-000001', 'FIC-TOL-A1', 'Good', 'Available'),
(2, 2, 'BK-000002', 'FIC-ORW-A2', 'Fair', 'Available'),
(3, 3, 'BK-000003', 'TEC-DAT-B1', 'New', 'Available'),
(4, 4, 'MAG-HIS-C1', 'MAGAZINE-C1', 'Good', 'Available'),
(5, 5, 'JRN-SCI-C2', 'JOURNAL-C2', 'Good', 'Available'),
(6, 6, 'REC-JAZ-D1', 'RECORD-D1', 'Fair', 'Available'),
(7, 7, 'LAP-TECH-01', 'TECH-DESK', 'Good', 'Available'),
(8, 8, 'CHR-USBC-01', 'SERVICE-DESK', 'New', 'Available'),
(9, 9, 'MRK-EXPO-01', 'EVENT-CABINET', 'Good', 'Available'),
(10, 10, 'ERS-QRT-01', 'EVENT-CABINET', 'Damaged', 'Available'),
(11, 12, 'BK-000004', 'FIC-HER-A3', 'Good', 'Available'),
(12, 13, 'BK-000005', 'FIC-LEG-A4', 'Good', 'Available'),
(13, 14, 'BK-000006', 'SELF-HELP-B2', 'New', 'Available'),
(14, 1, 'BK-000007', 'FIC-TOL-A1', 'Good', 'Available'),
(15, 7, 'LAP-TECH-02', 'TECH-REPAIR', 'Fair', 'Maintenance'),
(16, 12, 'BK-000008', 'REPAIR-CART', 'Damaged', 'Maintenance'),
(17, 2, 'BK-000009', 'LOST-RECORDS', 'Fair', 'Lost'),
(18, 15, 'REC-JAZ-D2', 'REC-JAZ-D1', 'Good', 'Available'),
(19, 4, 'MAG-HIS-C2', 'MAGAZINE-C1', 'New', 'Available'),
(20, 5, 'JRN-SCI-C3', 'JOURNAL-C2', 'Fair', 'Available'),
(21, 7, 'LAP-TECH-03', 'TECH-DESK', 'New', 'Available'),
(22, 8, 'CHR-USBC-02', 'SERVICE-DESK', 'Good', 'Available'),
(23, 9, 'MRK-EXPO-02', 'EVENT-CABINET', 'New', 'Available'),
(24, 10, 'ERS-QRT-02', 'EVENT-CABINET', 'Good', 'Available'),
(25, 15, 'REC-JAZ-D3', 'REC-JAZ-D1', 'New', 'Available');


Running query in 'sqlite:///library.db'

25 rows affected.

++
||
++
++

### Creating varied loans for every physical material type, then recording returns


In [45]:
%%sql
-- Loans begin open; return updates run the fine and item-status triggers.
INSERT INTO Loan (loanID, memberID, borrowableItemID, employeeID, borrowDateTime, dueDateTime, returnDateTime) VALUES
(1, 1, 1, 1, '2026-01-01 10:00', '2026-01-22 10:00', NULL),
(2, 2, 2, 4, '2026-01-03 14:30', '2026-01-24 14:30', NULL),
(3, 3, 3, 5, '2026-01-05 09:15', '2026-01-26 09:15', NULL),
(4, 6, 4, 1, '2026-01-07 12:00', '2026-01-28 12:00', NULL),
(5, 7, 5, 4, '2026-01-09 16:45', '2026-01-30 16:45', NULL),
(6, 8, 6, 5, '2026-01-11 11:20', '2026-02-01 11:20', NULL),
(7, 11, 7, 1, '2026-02-01 09:00', '2026-02-02 09:00', NULL),
(8, 12, 8, 4, '2026-02-03 10:00', '2026-02-03 14:00', NULL),
(9, 1, 9, 5, '2026-02-04 09:00', '2026-02-04 13:00', NULL),
(10, 2, 10, 1, '2026-02-05 12:00', '2026-02-05 16:00', NULL),
(11, 11, 18, 4, '2026-03-01 13:00', '2026-03-22 13:00', NULL),
(12, 6, 19, 1, '2026-04-01 10:00', '2026-04-22 10:00', NULL),
(13, 7, 20, 4, '2026-04-02 09:00', '2026-04-23 09:00', NULL),
(14, 8, 21, 5, '2026-04-03 11:00', '2026-04-04 11:00', NULL),
(15, 12, 22, 1, '2026-04-04 13:00', '2026-04-04 17:00', NULL),
(16, 1, 23, 4, '2026-04-05 14:00', '2026-04-05 18:00', NULL),
(17, 2, 24, 5, '2026-04-06 10:00', '2026-04-06 14:00', NULL),
(18, 3, 25, 1, '2026-04-07 12:00', '2026-04-28 12:00', NULL),
(19, 6, 11, 4, '2026-05-01 10:00', '2026-05-22 10:00', NULL),
(20, 7, 12, 5, '2026-05-02 11:00', '2026-05-23 11:00', NULL),
(21, 8, 13, 1, '2026-05-03 12:00', '2026-05-24 12:00', NULL);

UPDATE Loan SET returnDateTime='2026-01-24 10:00' WHERE loanID=1;
UPDATE Loan SET returnDateTime='2026-01-25 14:30' WHERE loanID=2;
UPDATE Loan SET returnDateTime='2026-01-29 09:15' WHERE loanID=3;
UPDATE Loan SET returnDateTime='2026-01-29 12:00' WHERE loanID=4;
UPDATE Loan SET returnDateTime='2026-02-01 16:45' WHERE loanID=5;
UPDATE Loan SET returnDateTime='2026-02-02 11:20' WHERE loanID=6;
UPDATE Loan SET returnDateTime='2026-02-02 11:00' WHERE loanID=7;
UPDATE Loan SET returnDateTime='2026-02-03 15:30' WHERE loanID=8;
UPDATE Loan SET returnDateTime='2026-02-04 15:00' WHERE loanID=9;
UPDATE Loan SET returnDateTime='2026-02-05 18:00' WHERE loanID=10;
UPDATE Loan SET returnDateTime='2026-04-20 10:00' WHERE loanID=12;
UPDATE Loan SET returnDateTime='2026-04-23 09:00' WHERE loanID=13;
UPDATE Loan SET returnDateTime='2026-04-04 07:00' WHERE loanID=14;
UPDATE Loan SET returnDateTime='2026-04-04 16:00' WHERE loanID=15;
UPDATE Loan SET returnDateTime='2026-04-05 18:00' WHERE loanID=16;
UPDATE Loan SET returnDateTime='2026-04-06 15:00' WHERE loanID=17;
UPDATE Loan SET returnDateTime='2026-05-25 10:00' WHERE loanID=19;
UPDATE Loan SET returnDateTime='2026-05-24 11:00' WHERE loanID=20;


Running query in 'sqlite:///library.db'

21 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

...

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

++
||
++
++

### Updating the fines created by the late-return trigger


In [46]:
%%sql
-- Amounts and issue dates are generated and protected by the database.
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-01-25', description='Two-day overdue print book' WHERE loanID=1;
UPDATE Fine SET paymentStatus='Waived', paymentDate=NULL, description='Delay waived after weather closure' WHERE loanID=2;
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-01-30', description='Three-day overdue database textbook' WHERE loanID=3;
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-01-30', description='One-day overdue magazine' WHERE loanID=4;
UPDATE Fine SET paymentStatus='Waived', paymentDate=NULL, description='Return-bin issue, fine waived' WHERE loanID=5;
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-02-03', description='One-day overdue jazz record' WHERE loanID=6;
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-02-03', description='Laptop returned two hours late' WHERE loanID=7;
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-02-04', description='USB-C charger returned late' WHERE loanID=8;
UPDATE Fine SET paymentStatus='Waived', paymentDate=NULL, description='Marker delay waived by desk staff' WHERE loanID=9;
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-02-06', description='Damaged eraser returned late' WHERE loanID=10;
UPDATE Fine SET paymentStatus='Paid', paymentDate='2026-04-07', description='Eraser returned one hour late' WHERE loanID=17;
UPDATE Fine SET paymentStatus='Unpaid', paymentDate=NULL, description='Dune returned three days late' WHERE loanID=19;
UPDATE Fine SET paymentStatus='Waived', paymentDate=NULL, description='One-day book delay waived after illness' WHERE loanID=20;


Running query in 'sqlite:///library.db'

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

...

1 rows affected.

1 rows affected.

1 rows affected.

1 rows affected.

++
||
++
++

### Inserting different kinds of library rooms


In [47]:
%%sql
INSERT INTO Room (roomID, roomName, floor, roomType, maximumCapacity) VALUES
(1, 'Cedar Reading Room', 1, 'Meeting Room', 24),
(2, 'Harbour Auditorium', 1, 'Activity Room', 180),
(3, 'Digital Learning Lab', 2, 'Study Room', 30),
(4, 'Rooftop Garden Room', 4, 'Activity Room', 45),
(5, 'Coast Salish Gallery', 1, 'Exhibition Room', 70),
(6, 'Maple Meeting Room', 2, 'Meeting Room', 16),
(7, 'Pacific Screening Room', 3, 'Activity Room', 95),
(8, 'Youth Gaming Room', 1, 'Gaming Room', 22),
(9, 'Courtyard Program Room', 1, 'Activity Room', 120),
(10, 'Community Art Gallery', 2, 'Exhibition Room', 55);


Running query in 'sqlite:///library.db'

10 rows affected.

++
||
++
++

### Inserting events in the required staging state


In [48]:
%%sql
-- SQLite does not allow an Open event before it has an organizer. Events
-- that will open are staged as Closed; an already-cancelled event can be
-- inserted as Cancelled. Final statuses are shown after registrations.
INSERT INTO Event (eventID, eventName, eventType, description, eventDate, startTime, endTime, roomID, eventCapacity, eventStatus) VALUES
(1, 'Dune Readers Circle', 'Book Club', 'Discussion of Frank Herbert''s Dune.', '2026-07-12', '10:00', '11:30', 1, 20, 'Closed'),
(2, 'An Evening with Eden Robinson', 'Author Talk', 'Canadian literature and writing.', '2026-07-18', '18:30', '20:00', 2, 150, 'Closed'),
(3, 'Build Your First SQLite Database', 'Book Workshop', 'Hands-on database workshop.', '2026-07-25', '13:00', '16:00', 3, 28, 'Closed'),
(4, 'Emerging Vancouver Artists', 'Art Show', 'Work by local artists.', '2026-08-01', '11:00', '17:00', 5, 60, 'Cancelled'),
(5, 'Canadian Cinema Night', 'Film Screening', 'Film screening and discussion.', '2026-08-06', '18:30', '21:00', 7, 90, 'Closed'),
(6, 'Languages of Vancouver Festival', 'Cultural Festival', 'Language, food, and stories.', '2026-08-07', '10:00', '16:00', 9, 110, 'Closed'),
(7, 'Teen Switch Tournament', 'Gaming Event', 'Friendly teen tournament.', '2026-08-08', '13:00', '16:00', 8, 20, 'Closed'),
(8, 'Neighbourhood Climate Action Meeting', 'Group Meeting', 'Public planning meeting.', '2026-08-15', '18:00', '19:30', 6, 16, 'Closed'),
(9, 'Autumn Family Story Picnic', 'Book Club', 'Family reading and stories.', '2026-08-22', '11:00', '13:00', 4, 40, 'Closed'),
(10, 'Documentary Photography Workshop', 'Book Workshop', 'Visual storytelling workshop.', '2026-09-05', '17:30', '20:00', 10, 45, 'Closed');


Running query in 'sqlite:///library.db'

10 rows affected.

++
||
++
++

### Assigning authorized organizers and opening events


In [49]:
%%sql
INSERT INTO OrganizedBy (employeeID, eventID) VALUES
(3, 1),
(1, 1),
(7, 2),
(1, 3),
(5, 4),
(3, 4),
(7, 5),
(3, 6),
(6, 6),
(5, 7),
(2, 8),
(7, 9),
(3, 10);

-- All events are opened after an organizer exists. Past and cancelled
-- events receive their final statuses after registrations are inserted.
UPDATE Event SET eventStatus='Open' WHERE eventID IN (1,2,3,5,6,7,8,9,10);


Running query in 'sqlite:///library.db'

13 rows affected.

9 rows affected.

++
||
++
++

### Inserting varied audience groups


In [50]:
%%sql
INSERT INTO Audience (audienceID, audienceName) VALUES
(1, 'Children'),
(2, 'Teenagers'),
(3, 'Adults'),
(4, 'Seniors'),
(5, 'Families'),
(6, 'Newcomers'),
(7, 'Students'),
(8, 'Educators'),
(9, 'Local Artists'),
(10, 'Technology Beginners'),
(11, 'Young Adults'),
(12, 'Researchers'),
(13, 'Parents and Caregivers'),
(14, 'Indigenous Communities'),
(15, 'French Speakers'),
(16, 'Job Seekers'),
(17, 'People with Disabilities'),
(18, 'Film Enthusiasts'),
(19, 'New Readers'),
(20, 'Small Business Owners');


Running query in 'sqlite:///library.db'

20 rows affected.

++
||
++
++

### Associating events with multiple audiences


In [51]:
%%sql
INSERT INTO EventAudience (eventID, audienceID) VALUES
(1, 3),
(1, 4),
(2, 3),
(2, 7),
(3, 7),
(3, 10),
(4, 9),
(5, 3),
(6, 5),
(6, 6),
(7, 2),
(8, 3),
(9, 1),
(9, 5),
(10, 8),
(1, 11),
(1, 19),
(2, 14),
(3, 16),
(4, 17),
(5, 18),
(6, 14),
(6, 15),
(7, 11),
(8, 12),
(8, 20),
(9, 13),
(9, 19),
(10, 17);


Running query in 'sqlite:///library.db'

29 rows affected.

++
||
++
++

### Associating events with varied interests


In [52]:
%%sql
INSERT INTO EventInterest (eventID, interestID) VALUES
(1, 2),
(1, 1),
(2, 1),
(3, 4),
(4, 7),
(5, 10),
(6, 3),
(6, 11),
(7, 12),
(7, 4),
(8, 9),
(9, 8),
(9, 9),
(10, 7),
(10, 10);


Running query in 'sqlite:///library.db'

15 rows affected.

++
||
++
++

### Associating people with varied interests


In [53]:
%%sql
INSERT INTO PersonInterest (personID, interestID) VALUES
(1, 2),
(1, 9),
(2, 3),
(3, 4),
(3, 7),
(4, 5),
(5, 6),
(6, 10),
(7, 12),
(7, 4),
(8, 1),
(9, 8),
(10, 11),
(11, 3),
(12, 9),
(23, 7),
(24, 10),
(25, 6),
(26, 3),
(27, 8);


Running query in 'sqlite:///library.db'

20 rows affected.

++
||
++
++

### Inserting registrations with every registration status


In [54]:
%%sql
INSERT INTO EventRegistration (registrationID, personID, eventID, registrationDate, registrationStatus) VALUES
(1, 13, 1, '2026-06-20', 'Attended'),
(2, 14, 2, '2026-06-25', 'Attended'),
(3, 15, 3, '2026-07-02', 'No-show'),
(4, 16, 5, '2026-07-10', 'Cancelled'),
(5, 17, 5, '2026-07-20', 'Registered'),
(6, 18, 6, '2026-07-21', 'Registered'),
(7, 19, 9, '2026-07-25', 'Registered'),
(8, 20, 10, '2026-07-28', 'Registered'),
(9, 21, 1, '2026-06-28', 'No-show'),
(10, 22, 2, '2026-07-01', 'Cancelled'),
(11, 23, 3, '2026-07-05', 'Attended'),
(12, 24, 6, '2026-07-22', 'Registered');

-- Apply final statuses after registration history exists. Event 4 was
-- already Cancelled; cancelling event 9 also cancels its registration.
UPDATE Event SET eventStatus='Closed' WHERE eventID IN (1,2,3);
UPDATE Event SET eventStatus='Cancelled' WHERE eventID=9;

-- Display the actual final Event table statuses after staging is complete.
SELECT eventID, eventName, eventDate, eventStatus
FROM Event
ORDER BY eventDate, startTime;


Running query in 'sqlite:///library.db'

12 rows affected.

3 rows affected.

1 rows affected.

eventID,eventName,eventDate,eventStatus
1,Dune Readers Circle,2026-07-12,Closed
2,An Evening with Eden Robinson,2026-07-18,Closed
3,Build Your First SQLite Database,2026-07-25,Closed
4,Emerging Vancouver Artists,2026-08-01,Cancelled
5,Canadian Cinema Night,2026-08-06,Open
6,Languages of Vancouver Festival,2026-08-07,Open
7,Teen Switch Tournament,2026-08-08,Open
8,Neighbourhood Climate Action Meeting,2026-08-15,Open
9,Autumn Family Story Picnic,2026-08-22,Cancelled
10,Documentary Photography Workshop,2026-09-05,Open


### Inserting volunteer assignments with every role and status


In [55]:
%%sql
INSERT INTO VolunteerAssignment (assignmentID, role, personID, eventID, status, numberOfHours) VALUES
(1, 'Registration Assistant', 23, 1, 'Completed', 2),
(2, 'Room Setup', 24, 2, 'Completed', 2.5),
(3, 'Event Guide', 25, 3, 'Completed', 3),
(4, 'Cleanup Assistant', 26, 4, 'Rejected', 1.5),
(5, 'Registration Assistant', 27, 5, 'Approved', 2),
(6, 'Event Guide', 28, 6, 'Approved', 4),
(7, 'Room Setup', 13, 7, 'Pending', 2),
(8, 'Cleanup Assistant', 14, 8, 'Approved', 1),
(9, 'Registration Assistant', 15, 9, 'Rejected', 2.5),
(10, 'Event Guide', 16, 10, 'Approved', 3.5),
(11, 'Cleanup Assistant', 17, 1, 'Completed', 1.5),
(12, 'Room Setup', 18, 6, 'Pending', 3);


Running query in 'sqlite:///library.db'

12 rows affected.

++
||
++
++

### Inserting donations in the required Pending state


In [56]:
%%sql
-- Items are added before reviewed donations are finalized.
INSERT INTO Donation (donationID, donorPersonID, reviewEmployeeID, donationDate, donationStatus) VALUES
(1, 1, 1, '2026-03-02', 'Pending'),
(2, 2, NULL, '2026-03-05', 'Pending'),
(3, 3, 2, '2026-03-08', 'Pending'),
(4, 4, 5, '2026-03-12', 'Pending'),
(5, 5, 2, '2026-03-18', 'Pending'),
(6, 6, 6, '2026-03-22', 'Pending'),
(7, 7, 1, '2026-04-01', 'Pending'),
(8, 8, 5, '2026-04-09', 'Pending'),
(9, 9, 2, '2026-04-16', 'Pending'),
(10, 10, NULL, '2026-04-25', 'Pending');


Running query in 'sqlite:///library.db'

10 rows affected.

++
||
++
++

### Inserting donated items and finalizing reviewed donations


In [57]:
%%sql
INSERT INTO DonatedItems (donatedItemID, donationID, title, itemType, author, quantity, itemCondition, approvalStatus) VALUES
(1, 1, 'The Left Hand of Darkness', 'Print Book', 'Ursula K. Le Guin', 1, 'Good', 'Approved'),
(2, 1, 'Calculus Practice Workbook', 'Print Book', 'Various Authors', 1, 'Damaged', 'Rejected'),
(3, 2, 'Canadian Living - 2025 Issues', 'Magazine', 'Canadian Living Editors', 8, 'Good', 'Pending'),
(4, 3, 'World Encyclopedia - 1998 Edition', 'Print Book', 'Encyclopedia Editors', 12, 'Fair', 'Rejected'),
(5, 4, 'Blue Note Jazz Collection', 'Record', 'Various Artists', 6, 'Good', 'Approved'),
(6, 4, 'The Jazz Standards', 'Print Book', 'Ted Gioia', 1, 'New', 'Approved'),
(7, 5, 'French Easy Readers', 'Print Book', 'Various Authors', 5, 'Good', 'Pending'),
(8, 6, '2012 Notebook Computer', 'Laptop', 'Various', 1, 'Damaged', 'Rejected'),
(9, 7, 'Canadian Picture Book Set', 'Print Book', 'Various Canadian Authors', 10, 'New', 'Approved'),
(10, 7, 'Assorted Permanent Markers', 'Marker', 'Various', 14, 'Fair', 'Rejected'),
(11, 8, 'Dune', 'Print Book', 'Frank Herbert', 2, 'Good', 'Approved'),
(12, 8, 'Classical Piano Collection', 'Record', 'Various Artists', 4, 'Fair', 'Approved'),
(13, 9, 'Mixed Phone Chargers', 'Charger', 'Various', 9, 'Damaged', 'Rejected'),
(14, 10, 'The Complete Canadian Cookbook', 'Print Book', 'Various Authors', 1, 'Good', 'Pending'),
(15, 10, 'Cycling Maps of British Columbia', 'Map', 'BC Cycling Coalition', 7, NULL, 'Pending');

UPDATE Donation SET donationStatus='Approved' WHERE donationID IN (1,4,7,8);
UPDATE Donation SET donationStatus='Rejected' WHERE donationID IN (3,6,9);


Running query in 'sqlite:///library.db'

15 rows affected.

4 rows affected.

3 rows affected.

++
||
++
++

### Inserting acquisition requests with every request status


In [58]:
%%sql
INSERT INTO FutureAcquisitions (acquisitionID, title, itemType, author, proposedQuantity, estimatedCost, requestDate, requestStatus, personID, employeeID) VALUES
(1, 'Braiding Sweetgrass', 'Print Book', 'Robin Wall Kimmerer', 3, 66, '2026-05-01', 'Proposed', 1, NULL),
(2, 'Epson PowerLite Projector', 'Equipment', 'Epson', 1, 899, '2026-05-03', 'Under Review', 2, 1),
(3, 'Kobo Clara Colour eReaders', 'Technology', 'Rakuten Kobo', 6, 899.94, '2026-05-05', 'Approved', 3, 2),
(4, 'Meta Quest VR Headsets', 'Technology', 'Meta', 4, 2399.96, '2026-05-08', 'Rejected', 4, 5),
(5, 'Oxford Spanish Dictionary', 'Print Book', 'Oxford Languages', 4, NULL, '2026-05-10', 'Proposed', 5, NULL),
(6, 'LEGO Education Robotics Kits', 'Technology Kit', 'LEGO Education', 8, 1600, '2026-05-13', 'Under Review', 6, 6),
(7, 'MacBook Air Laptops', 'Laptop', 'Apple', 5, 6495, '2026-05-15', 'Approved', 7, 1),
(8, 'Vinyl Record Cleaning Machine', 'Equipment', 'Pro-Ject', 1, 799, '2026-05-18', 'Rejected', 8, 2),
(9, 'Penguin Classics eBook Package', 'Online Book', 'Penguin Random House', 1, 1200, '2026-05-20', 'Approved', 9, 5),
(10, 'Modern Board Game Collection', 'Gaming Equipment', 'Various Designers', 12, 900, '2026-05-22', 'Under Review', 10, 6);

INSERT INTO AddCollection
(
    sourceType,
    sourceID,
    libraryItemID,
    processEmployeeID,
    processAt
)
VALUES
(
    'Acquisition',
    9,
    21,
    5,
    '2026-05-23 10:30'
);



Running query in 'sqlite:///library.db'

10 rows affected.

1 rows affected.

++
||
++
++

## 6. Confirm that every table has at least 10 tuples


In [59]:
%%sql
WITH tableCounts(tableName, tupleCount) AS (
    SELECT 'Person', COUNT(*) FROM Person
    UNION ALL SELECT 'Member', COUNT(*) FROM Member
    UNION ALL SELECT 'Interest', COUNT(*) FROM Interest
    UNION ALL SELECT 'Employee', COUNT(*) FROM Employee
    UNION ALL SELECT 'LibraryItem', COUNT(*) FROM LibraryItem
    UNION ALL SELECT 'BorrowableItem', COUNT(*) FROM BorrowableItem
    UNION ALL SELECT 'Loan', COUNT(*) FROM Loan
    UNION ALL SELECT 'Fine', COUNT(*) FROM Fine
    UNION ALL SELECT 'Room', COUNT(*) FROM Room
    UNION ALL SELECT 'Event', COUNT(*) FROM Event
    UNION ALL SELECT 'OrganizedBy', COUNT(*) FROM OrganizedBy
    UNION ALL SELECT 'Audience', COUNT(*) FROM Audience
    UNION ALL SELECT 'EventAudience', COUNT(*) FROM EventAudience
    UNION ALL SELECT 'EventInterest', COUNT(*) FROM EventInterest
    UNION ALL SELECT 'PersonInterest', COUNT(*) FROM PersonInterest
    UNION ALL SELECT 'EventRegistration', COUNT(*) FROM EventRegistration
    UNION ALL SELECT 'VolunteerAssignment', COUNT(*) FROM VolunteerAssignment
    UNION ALL SELECT 'Donation', COUNT(*) FROM Donation
    UNION ALL SELECT 'DonatedItems', COUNT(*) FROM DonatedItems
    UNION ALL SELECT 'FutureAcquisitions', COUNT(*) FROM FutureAcquisitions
)
SELECT
    tableName,
    tupleCount,
    CASE
        WHEN tupleCount >= 10 THEN 'PASS'
        ELSE 'NEEDS MORE DATA'
    END AS requirementCheck
FROM tableCounts
ORDER BY tableName;


Running query in 'sqlite:///library.db'

tableName,tupleCount,requirementCheck
Audience,20,PASS
BorrowableItem,25,PASS
DonatedItems,15,PASS
Donation,10,PASS
Employee,10,PASS
Event,10,PASS
EventAudience,29,PASS
EventInterest,15,PASS
EventRegistration,12,PASS
Fine,13,PASS


## 7. Check database and foreign-key integrity


In [60]:
%%sql
SELECT
    'Database integrity' AS checkName,
    integrity_check AS result
FROM pragma_integrity_check

UNION ALL

SELECT
    'Foreign key violations' AS checkName,
    CASE
        WHEN COUNT(*) = 0 THEN '0 - PASS'
        ELSE CAST(COUNT(*) AS TEXT) || ' - CHECK REQUIRED'
    END AS result
FROM pragma_foreign_key_check;


Running query in 'sqlite:///library.db'

checkName,result
Database integrity,ok
Foreign key violations,0 - PASS


## 8. Review the database objects created in Sections 3 and 4


In [61]:
%%sql
SELECT
    type AS objectType,
    name AS objectName,
    tbl_name AS relatedTable
FROM sqlite_master
WHERE type IN ('table', 'view', 'index', 'trigger')
  AND name NOT LIKE 'sqlite_%'
ORDER BY
    CASE type
        WHEN 'table' THEN 1
        WHEN 'view' THEN 2
        WHEN 'index' THEN 3
        WHEN 'trigger' THEN 4
    END,
    name;


Running query in 'sqlite:///library.db'

objectType,objectName,relatedTable
table,AddCollection,AddCollection
table,Audience,Audience
table,Auth,Auth
table,BorrowableItem,BorrowableItem
table,DonatedItems,DonatedItems
table,Donation,Donation
table,Employee,Employee
table,Event,Event
table,EventAudience,EventAudience
table,EventInterest,EventInterest


## 9. View the item collection and physical-copy availability


In [62]:
%%sql
SELECT
    li.itemID,
    li.itemTitle,
    li.itemType,
    COUNT(bi.borrowableItemID) AS totalPhysicalCopies,
    SUM(CASE WHEN bi.itemStatus = 'Available' THEN 1 ELSE 0 END) AS availableCopies,
    SUM(CASE WHEN bi.itemStatus = 'Borrowed' THEN 1 ELSE 0 END) AS borrowedCopies,
    SUM(CASE WHEN bi.itemStatus = 'Maintenance' THEN 1 ELSE 0 END) AS maintenanceCopies,
    SUM(CASE WHEN bi.itemStatus = 'Lost' THEN 1 ELSE 0 END) AS lostCopies,
    CASE
        WHEN li.itemType = 'Online Book' THEN 'Online access'
        WHEN COUNT(bi.borrowableItemID) = 0 THEN 'No physical copy'
        ELSE 'Physical material'
    END AS accessType
FROM LibraryItem li
LEFT JOIN BorrowableItem bi
    ON bi.itemID = li.itemID
GROUP BY li.itemID, li.itemTitle, li.itemType
ORDER BY li.itemType, li.itemTitle;


Running query in 'sqlite:///library.db'

itemID,itemTitle,itemType,totalPhysicalCopies,availableCopies,borrowedCopies,maintenanceCopies,lostCopies,accessType
8,Anker 735 USB-C Charger,Charger,2,2,0,0,0,Physical material
10,Quartet Whiteboard Eraser,Eraser,2,1,0,1,0,Physical material
7,Dell Latitude 5440,Laptop,3,2,0,1,0,Physical material
4,NASA Spinoff 2024,Magazine,2,2,0,0,0,Physical material
9,Expo Low Odor Marker Set,Marker,2,2,0,0,0,Physical material
19,Alice's Adventures in Wonderland,Online Book,0,0,0,0,0,Online access
18,Anne of Green Gables,Online Book,0,0,0,0,0,Online access
16,Frankenstein,Online Book,0,0,0,0,0,Online access
17,Little Women,Online Book,0,0,0,0,0,Online access
21,Penguin Classics eBook Package,Online Book,0,0,0,0,0,Online access


## 10. View the event schedule, organizers, audiences, and registration totals


In [63]:
%%sql
SELECT
    e.eventID,
    e.eventName,
    e.eventType,
    e.eventDate,
    e.startTime || ' - ' || e.endTime AS eventTime,
    r.roomName,
    e.eventCapacity,
    e.eventStatus,

    (
        SELECT GROUP_CONCAT(p.firstName || ' ' || p.lastName, ', ')
        FROM OrganizedBy ob
        JOIN Employee emp
            ON emp.employeeID = ob.employeeID
        JOIN Person p
            ON p.personID = emp.personID
        WHERE ob.eventID = e.eventID
    ) AS organizers,

    (
        SELECT GROUP_CONCAT(a.audienceName, ', ')
        FROM EventAudience ea
        JOIN Audience a
            ON a.audienceID = ea.audienceID
        WHERE ea.eventID = e.eventID
    ) AS targetAudiences,

    (
        SELECT COUNT(*)
        FROM EventRegistration er
        WHERE er.eventID = e.eventID
          AND er.registrationStatus = 'Registered'
    ) AS registeredCount

FROM Event e
JOIN Room r
    ON r.roomID = e.roomID
ORDER BY e.eventDate, e.startTime;


Running query in 'sqlite:///library.db'

eventID,eventName,eventType,eventDate,eventTime,roomName,eventCapacity,eventStatus,organizers,targetAudiences,registeredCount
1,Dune Readers Circle,Book Club,2026-07-12,10:00 - 11:30,Cedar Reading Room,20,Closed,"Priya Patel, Emily Carter","Adults, Seniors, Young Adults, New Readers",0
2,An Evening with Eden Robinson,Author Talk,2026-07-18,18:30 - 20:00,Harbour Auditorium,150,Closed,Isabella Brown,"Adults, Students, Indigenous Communities",0
3,Build Your First SQLite Database,Book Workshop,2026-07-25,13:00 - 16:00,Digital Learning Lab,28,Closed,Emily Carter,"Students, Technology Beginners, Job Seekers",0
4,Emerging Vancouver Artists,Art Show,2026-08-01,11:00 - 17:00,Coast Salish Gallery,60,Cancelled,"Maya Thompson, Priya Patel","Local Artists, People with Disabilities",0
5,Canadian Cinema Night,Film Screening,2026-08-06,18:30 - 21:00,Pacific Screening Room,90,Open,Isabella Brown,"Adults, Film Enthusiasts",1
6,Languages of Vancouver Festival,Cultural Festival,2026-08-07,10:00 - 16:00,Courtyard Program Room,110,Open,"Priya Patel, Gabriel Nakamura","Families, Newcomers, Indigenous Communities, French Speakers",2
7,Teen Switch Tournament,Gaming Event,2026-08-08,13:00 - 16:00,Youth Gaming Room,20,Open,Maya Thompson,"Teenagers, Young Adults",0
8,Neighbourhood Climate Action Meeting,Group Meeting,2026-08-15,18:00 - 19:30,Maple Meeting Room,16,Open,Daniel Kim,"Adults, Researchers, Small Business Owners",0
9,Autumn Family Story Picnic,Book Club,2026-08-22,11:00 - 13:00,Rooftop Garden Room,40,Cancelled,Isabella Brown,"Children, Families, Parents and Caregivers, New Readers",0
10,Documentary Photography Workshop,Book Workshop,2026-09-05,17:30 - 20:00,Community Art Gallery,45,Open,Priya Patel,"Educators, People with Disabilities",1
